# 22. 구조 기반 청킹 + 제목 경로 검색문

이 노트북은 Codex coder agent가 작성하고 실행한 개발셋 전용 오프라인 preflight입니다. 순수 경계 ablation이 아니라 raw TXT의 페이지 표식과 Markdown H1~H6 구조로 direct body 청크를 만들고, issuer/card/제목 경로를 retrieval_text에 붙이는 결합 실험입니다.

이번 실행은 구조 청크 생성, 감사, 캐시 재사용 판정, 향후 embedding 전송 계획까지만 수행합니다. API key를 읽지 않고, 네트워크·API 요청·새 embedding·Chroma query를 수행하지 않습니다. 승인 이후의 embedding과 검색 평가는 이 노트북의 후속 실행 경계 밖입니다.

## 결과 계산 전에 고정한 계약

- 입력은 명시한 raw TXT 10개, 기존 개발 query text 30개, 기존 embedding cache 6개와 읽기 전용 기준 파일뿐입니다.
- structured JSON과 benefit/numeric label은 청킹·검색 입력에 사용하지 않습니다. expected card/level/required terms/category는 향후 ranking Top50을 먼저 고정하고 hash한 뒤 평가에만 사용합니다.
- heading stack은 페이지를 넘어 유지합니다. 새 heading은 같은 level 이하를 교체하고, 건너뛴 level은 가장 가까운 상위 heading 아래에 연결합니다. 제목 전 서문은 root direct body입니다.
- 부모 direct body에 자식 body를 복제하지 않습니다. heading-only node는 hierarchy manifest에만 남습니다.
- direct body가 4,000자를 넘으면 문단 경계, 실패 시 줄 경계로 나눕니다. overlap, truncate, 문자 중간 분할은 없습니다.
- retrieval_text는 issuer/card_name + heading_path + body입니다. evidence_text는 heading_path + body이며 인위적으로 붙인 issuer/card_name을 제외합니다.
- 향후 검색은 NumPy squared-L2와 chunk_id lexical tie, normalized BM25(k1=1.5, b=0.75), RRF vector:BM25=0.4:0.6, k=60, depth=50입니다. reranker/MMR/morphology/router는 사용하지 않습니다.
- 기존 327개 기준선도 NumPy exact로 다시 만들고 공개 Top5와 16번 0.4 결과를 재현합니다. 불일치 시 새 결과를 exact-v1으로 별도 명명합니다.
- 향후 평가 ranking은 gold를 읽기 전에 Top50과 hash를 고정합니다. Card 10은 Card Hit@3/Card MRR@5, Evidence 20은 expected card + required terms가 evidence_text에 모두 존재하는 level_relaxed_term_bearing 기준입니다.
- heading만으로 required term을 충족하는 부풀림을 별도 감사하고 relevant set을 수동 검토 파일로 저장합니다.
- primary gate는 무결성 PASS, Evidence Hit@3 비회귀, 전체 Card Hit@3 비회귀, 그리고 Evidence Hit@3 최소 1질의 개선 또는 Hit 동률에서 MRR@5 delta >= 0.025 및 paired wins > losses입니다.
- body 기준 same-card exact duplicate와 NFKC/lower/공백·토큰 정규화한 짧은 body 80% containment가 모두 비악화하고 하나는 개선해야 합니다. Recall/nDCG는 청크 분모 영향 때문에 진단만 합니다.
- numeric/semantic 분리, win/loss/tie, changed Top5를 저장합니다. gate 통과도 다음 reranker 재평가 후보일 뿐 운영 또는 holdout 승격이 아닙니다.

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)


In [2]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import statistics
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import tiktoken

ROOT = Path.cwd()
assert ROOT.name == "PickCardU"

OUTPUT_DIR = ROOT / "notebooks/data/22_structural_heading_chunking_ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "구조 기반 청킹 + 제목 경로 검색문"
MODEL = "text-embedding-3-small"
MAX_BODY_CHARS = 4000
SHORT_BODY_CHARS = 80
BATCH_SIZE = 64
APPROVAL_CAPS = {"max_new_items": 1000, "max_input_tokens": 250000, "max_requests": 20}

RAW_SOURCES = [
    ("data/ocr_benchmark/gold/raw/BC/BC_Biz_AirMoney.txt", "02b515ced99a702101e3a02a5f44c7d91f59632148d1de1063ae28caf0615255"),
    ("data/ocr_benchmark/gold/raw/NH/NH_Namu_NH.txt", "6e8c461421516becf8cf4e1aeba36025396d9ddb50c30cc50588f0adf2fd30e7"),
    ("data/ocr_benchmark/gold/raw/hana/Hana_One_More_SOHO.txt", "139345f589e06492e2140b72e09abbb695b34b8510e226979cf154fd4e684598"),
    ("data/ocr_benchmark/gold/raw/hyundai/Hyundai_The_Orange_20260330.txt", "d3f7e14735e8f5f2677cc41c077ef66fd408976caabb4c8b1ed2a7af16026b0f"),
    ("data/ocr_benchmark/gold/raw/ibk/IBK_Point3.8(Credit).txt", "6684356a25bddc65d993380dc6e936f8f2285818fc8688cba35f4870b92d7e9f"),
    ("data/ocr_benchmark/gold/raw/kookmin/Kookmin_Friend_20210917.txt", "e3f38762b02d4b4e683d1738b320d292e55c07731c3d848850bee360fa622349"),
    ("data/ocr_benchmark/gold/raw/lotte/Lotte_LOCA_LIKIT_Eat.txt", "b3ba13a229d1c12df8675e581259e12d2696c9a5cc4b7d9ff5bb2c9ae7727cc3"),
    ("data/ocr_benchmark/gold/raw/samsung/Samsung_iD_ALL.txt", "a608b4ccca0d6b8e4f647b21d87b1c1e2f46c2e41d851d66f880856e759db48e"),
    ("data/ocr_benchmark/gold/raw/shinhan/Shinhan_Toss_Mr.Life_20251231.txt", "1024fe3732cff2c35559c1481ab5133f68d7594dc6edd15d679b8e2cfb7222ee"),
    ("data/ocr_benchmark/gold/raw/woori/Woori_Classic_EVERY_MILE_SKYPASS.txt", "9d9fad61bcf96df1090cc58aa7cb1b2f1b47336ae8d5bbd3bd1804679f76041c"),
]
QUERY_CSV = ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/retrieval_per_query.csv"
SOURCE_CHUNKS = ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl"
SOURCE_NOTEBOOK = ROOT / "notebooks/13_hierarchical_chunking_retrieval.ipynb"
SOURCE_INPUT_MANIFEST = ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/input_manifest.json"
SOURCE_EMBEDDING_USAGE = ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/embedding_usage.json"
SOURCE_16_CANDIDATES = ROOT / "notebooks/data/16_normalized_rrf_weight_ablation/rrf_weight_candidates.csv"

CACHE_FINGERPRINTS = [
    "3c81bda6bc8b1e69ca305b0dcc219203baeede6797e91045dbde36244babc3b4",
    "335a583c624bd1fe61aec74282af4e1462af20962c96147ada05ed5a29605fba",
    "93cd3f7a2ca8c9e070421705a575465a92f29966aebae38a583c2da60a6ee97f",
    "b6ae13c54bbba3d250e83c39da8673647b077dfc91f671565b4cef016593fbf2",
    "8433343533d6e8ae60f40c42d7e4a0b43b751bb36e26532e738ff51eac4a9ae0",
    "cb5d9e87e091be498d91fa51d46619ca60c7fe5314b27a6686b017cb8c304e55",
]
CACHE_FILES = [
    ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/embedding_cache/text-embedding-3-small" / (fingerprint + ".npz")
    for fingerprint in CACHE_FINGERPRINTS
]
CHROMA_FILES = [
    ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/chroma/2872d9ff-2f46-45c6-8731-f1f2c6a12757/data_level0.bin",
    ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/chroma/2872d9ff-2f46-45c6-8731-f1f2c6a12757/header.bin",
    ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/chroma/2872d9ff-2f46-45c6-8731-f1f2c6a12757/length.bin",
    ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/chroma/2872d9ff-2f46-45c6-8731-f1f2c6a12757/link_lists.bin",
    ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/chroma/chroma.sqlite3",
]

EVALUATION_CONTRACT = {
    "declared_before_results": True,
    "experiment_name": EXPERIMENT_NAME,
    "scope": "development_30_queries_only",
    "ranking_freeze_before_gold": {"depth": 50, "required": True, "sha256_required": True},
    "ranking_inputs_exclude": [
        "structured_json", "benefit_label", "numeric_label", "expected_card",
        "expected_level", "required_terms", "category",
    ],
    "vector": {
        "distance": "numpy_squared_l2",
        "tie_break": "chunk_id_lexical",
        "existing_corpus_count": 327,
        "baseline_reproduction": "public_top5_and_notebook16_vector_0.4_bm25_0.6",
        "mismatch_name": "exact-v1",
    },
    "bm25": {"normalization": "notebook13_normalized_search_tokens_exact", "k1": 1.5, "b": 0.75},
    "rrf": {"vector_weight": 0.4, "bm25_weight": 0.6, "k": 60, "depth": 50},
    "excluded_methods": ["reranker", "MMR", "morphology", "router"],
    "card10": ["Card Hit@3", "Card MRR@5"],
    "evidence20_relevance": "expected card and every required term occur in evidence_text; level ignored",
    "heading_inflation_audit": "separate heading-only term matches and write manual-review relevance sets",
    "primary_gate": {
        "integrity": "PASS",
        "evidence_hit_at_3": "non_regression",
        "all_card_hit_at_3": "non_regression",
        "improvement": "at least one evidence query Hit@3 improvement OR tied macro Hit@3 with MRR@5 delta >= 0.025 and paired wins > losses",
        "redundancy": "same-card body exact duplicates and normalized short-body 80% containment both non-worse and at least one improves",
    },
    "diagnostic_only": ["Recall", "nDCG"],
    "breakdowns": ["numeric", "semantic"],
    "promotion_limit": "reranker_reevaluation_candidate_only_not_operations_or_holdout",
}
print(json.dumps(EVALUATION_CONTRACT, ensure_ascii=False, indent=2))


{
  "declared_before_results": true,
  "experiment_name": "구조 기반 청킹 + 제목 경로 검색문",
  "scope": "development_30_queries_only",
  "ranking_freeze_before_gold": {
    "depth": 50,
    "required": true,
    "sha256_required": true
  },
  "ranking_inputs_exclude": [
    "structured_json",
    "benefit_label",
    "numeric_label",
    "expected_card",
    "expected_level",
    "required_terms",
    "category"
  ],
  "vector": {
    "distance": "numpy_squared_l2",
    "tie_break": "chunk_id_lexical",
    "existing_corpus_count": 327,
    "baseline_reproduction": "public_top5_and_notebook16_vector_0.4_bm25_0.6",
    "mismatch_name": "exact-v1"
  },
  "bm25": {
    "normalization": "notebook13_normalized_search_tokens_exact",
    "k1": 1.5,
    "b": 0.75
  },
  "rrf": {
    "vector_weight": 0.4,
    "bm25_weight": 0.6,
    "k": 60,
    "depth": 50
  },
  "excluded_methods": [
    "reranker",
    "MMR",
    "morphology",
    "router"
  ],
  "card10": [
    "Card Hit@3",
    "Card MRR@5"
  ],
  "ev

In [3]:
def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def raw_sha256(path: Path) -> str:
    return sha256_bytes(path.read_bytes())


def canonical_json(value: object) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def write_json(path: Path, value: object) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")


def write_csv(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def source_hash_map() -> dict[str, str]:
    explicit_paths = (
        [ROOT / path for path, _ in RAW_SOURCES]
        + [
            QUERY_CSV,
            SOURCE_CHUNKS,
            SOURCE_NOTEBOOK,
            SOURCE_INPUT_MANIFEST,
            SOURCE_EMBEDDING_USAGE,
            SOURCE_16_CANDIDATES,
        ]
        + CACHE_FILES
        + CHROMA_FILES
    )
    assert len(explicit_paths) == len(set(explicit_paths))
    return {str(path.relative_to(ROOT)): raw_sha256(path) for path in explicit_paths}


def text_sha256(text: str) -> str:
    return sha256_bytes(text.encode("utf-8"))


def rendered_char_count(records: list[dict]) -> int:
    return len("\n".join(record["text"] for record in records))


def split_records(records: list[dict], max_chars: int) -> tuple[list[list[dict]], str]:
    assert records and any(record["text"].strip() for record in records)
    paragraphs: list[list[dict]] = []
    current: list[dict] = []
    for record in records:
        current.append(record)
        if not record["text"].strip():
            paragraphs.append(current)
            current = []
    if current:
        paragraphs.append(current)

    units: list[list[dict]] = []
    used_line_fallback = False
    for paragraph in paragraphs:
        if rendered_char_count(paragraph) <= max_chars:
            units.append(paragraph)
            continue
        used_line_fallback = True
        line_group: list[dict] = []
        for record in paragraph:
            assert len(record["text"]) <= max_chars, "A source line exceeds max_chars; character splitting is forbidden."
            proposed = line_group + [record]
            if line_group and rendered_char_count(proposed) > max_chars:
                units.append(line_group)
                line_group = [record]
            else:
                line_group = proposed
        if line_group:
            units.append(line_group)

    parts: list[list[dict]] = []
    packed: list[dict] = []
    for unit in units:
        proposed = packed + unit
        if packed and rendered_char_count(proposed) > max_chars:
            parts.append(packed)
            packed = list(unit)
        else:
            packed = proposed
    if packed:
        parts.append(packed)

    assert all(rendered_char_count(part) <= max_chars for part in parts)
    assert [record["line_number"] for part in parts for record in part] == [
        record["line_number"] for record in records
    ]
    method = "line_boundary_fallback" if used_line_fallback else "paragraph_boundary"
    return parts, method


PAGE_RE = re.compile(r"^\[page\s*(\d+)\]\s*$", re.IGNORECASE)
HEADING_RE = re.compile(r"^\s{0,3}(#{1,6})\s+(.+?)\s*$")


def parse_card(path: Path, expected_sha: str) -> tuple[list[dict], list[dict], dict]:
    raw = path.read_text(encoding="utf-8")
    assert raw_sha256(path) == expected_sha
    issuer = path.parent.name
    card_name = path.stem
    card_key = issuer + "/" + card_name
    source_rel = str(path.relative_to(ROOT))
    source_hash = raw_sha256(path)

    root_id = card_key + "::node0000"
    nodes: list[dict] = [{
        "node_id": root_id,
        "parent_id": None,
        "heading_text": None,
        "heading_level": 0,
        "heading_path": [],
        "heading_line_number": None,
        "heading_page": None,
        "records": [],
    }]
    nodes_by_id = {root_id: nodes[0]}
    stack: list[dict] = []
    current_node = nodes[0]
    current_page = 1
    page_markers = 0
    headings = 0
    body_lines = 0
    assignments: dict[int, str] = {}

    lines = raw.splitlines()
    for line_number, line in enumerate(lines, start=1):
        page_match = PAGE_RE.fullmatch(line)
        if page_match:
            current_page = int(page_match.group(1))
            page_markers += 1
            assignments[line_number] = "page_marker"
            continue

        heading_match = HEADING_RE.fullmatch(line)
        if heading_match:
            level = len(heading_match.group(1))
            heading_text = heading_match.group(2).strip()
            while stack and stack[-1]["heading_level"] >= level:
                stack.pop()
            parent = stack[-1] if stack else nodes[0]
            node_id = card_key + "::node" + str(len(nodes)).zfill(4)
            heading_path = parent["heading_path"] + [heading_text]
            current_node = {
                "node_id": node_id,
                "parent_id": parent["node_id"],
                "heading_text": heading_text,
                "heading_level": level,
                "heading_path": heading_path,
                "heading_line_number": line_number,
                "heading_page": current_page,
                "records": [],
            }
            nodes.append(current_node)
            nodes_by_id[node_id] = current_node
            stack.append(current_node)
            headings += 1
            assignments[line_number] = "heading"
            continue

        current_node["records"].append({
            "line_number": line_number,
            "page": current_page,
            "text": line,
        })
        body_lines += 1
        assignments[line_number] = "direct_body"

    assert len(assignments) == len(lines)
    assert Counter(assignments.values()) == Counter({
        "page_marker": page_markers,
        "heading": headings,
        "direct_body": body_lines,
    })

    search_chunks: list[dict] = []
    hierarchy_rows: list[dict] = []
    covered_search_body_lines: list[int] = []
    for node in nodes:
        records = node["records"]
        substantive = bool(records and any(record["text"].strip() for record in records))
        parts: list[list[dict]] = []
        split_method = "not_applicable"
        if substantive:
            parts, split_method = split_records(records, MAX_BODY_CHARS)
        chunk_ids: list[str] = []
        for part_index, part in enumerate(parts, start=1):
            body = "\n".join(record["text"] for record in part)
            assert body.strip()
            heading_path = list(node["heading_path"])
            path_text = " > ".join(heading_path)
            retrieval_parts = [issuer, card_name]
            if path_text:
                retrieval_parts.append(path_text)
            retrieval_parts.append(body)
            evidence_parts = ([path_text] if path_text else []) + [body]
            retrieval_text = "\n".join(retrieval_parts)
            evidence_text = "\n".join(evidence_parts)
            assert retrieval_text.strip() and evidence_text.strip()
            assert not evidence_text.startswith(issuer + "\n" + card_name)
            pages = sorted({record["page"] for record in part})
            body_hash = text_sha256(body)
            chunk_seed = canonical_json({
                "experiment": EXPERIMENT_NAME,
                "source_hash": source_hash,
                "node_id": node["node_id"],
                "part_index": part_index,
                "body_sha256": body_hash,
            })
            chunk_id = "shc_" + text_sha256(chunk_seed)[:24]
            chunk_ids.append(chunk_id)
            covered_search_body_lines.extend(record["line_number"] for record in part)
            search_chunks.append({
                "chunk_id": chunk_id,
                "body": body,
                "heading_path": heading_path,
                "retrieval_text": retrieval_text,
                "evidence_text": evidence_text,
                "metadata": {
                    "issuer": issuer,
                    "card_name": card_name,
                    "card_key": card_key,
                    "page_numbers": pages,
                    "page_start": min(pages),
                    "page_end": max(pages),
                    "heading_level": node["heading_level"],
                    "node_id": node["node_id"],
                    "parent_id": node["parent_id"],
                    "part_index": part_index,
                    "part_count": len(parts),
                    "source_path": source_rel,
                    "source_sha256": source_hash,
                    "body_sha256": body_hash,
                    "split_method": split_method,
                },
            })

        node_pages = sorted(
            ({node["heading_page"]} if node["heading_page"] is not None else set())
            | {record["page"] for record in records}
        )
        hierarchy_rows.append({
            "issuer": issuer,
            "card_name": card_name,
            "card_key": card_key,
            "node_id": node["node_id"],
            "parent_id": node["parent_id"],
            "heading_text": node["heading_text"],
            "heading_level": node["heading_level"],
            "heading_path": node["heading_path"],
            "heading_line_number": node["heading_line_number"],
            "heading_page": node["heading_page"],
            "page_numbers": node_pages,
            "page_start": min(node_pages) if node_pages else None,
            "page_end": max(node_pages) if node_pages else None,
            "direct_body_line_count": len(records),
            "direct_body_nonblank_line_count": sum(bool(record["text"].strip()) for record in records),
            "direct_body_chars": rendered_char_count(records) if records else 0,
            "heading_only": not substantive,
            "search_chunk_ids": chunk_ids,
            "part_count": len(chunk_ids),
            "source_path": source_rel,
            "source_sha256": source_hash,
        })

    searchable_expected = sorted(
        record["line_number"]
        for node in nodes
        if node["records"] and any(record["text"].strip() for record in node["records"])
        for record in node["records"]
    )
    assert sorted(covered_search_body_lines) == searchable_expected
    assert len(covered_search_body_lines) == len(set(covered_search_body_lines))
    assert not any(row["heading_only"] and row["search_chunk_ids"] for row in hierarchy_rows)

    audit = {
        "source_path": source_rel,
        "source_sha256": source_hash,
        "issuer": issuer,
        "card_name": card_name,
        "card_key": card_key,
        "raw_chars": len(raw),
        "raw_lines": len(lines),
        "page_marker_lines": page_markers,
        "heading_lines": headings,
        "direct_body_lines": body_lines,
        "assigned_once_lines": len(assignments),
        "searchable_direct_body_lines": len(searchable_expected),
        "searchable_direct_body_lines_covered_once": len(covered_search_body_lines),
        "hierarchy_nodes": len(nodes),
        "heading_only_nodes": sum(row["heading_only"] for row in hierarchy_rows),
        "search_chunks": len(search_chunks),
    }
    return search_chunks, hierarchy_rows, audit


In [4]:
SOURCE_HASHES_BEFORE = source_hash_map()
for raw_rel, expected_sha in RAW_SOURCES:
    assert SOURCE_HASHES_BEFORE[raw_rel] == expected_sha

query_text_by_id: dict[str, str] = {}
with QUERY_CSV.open(encoding="utf-8", newline="") as handle:
    for row in csv.DictReader(handle):
        if row["method"] == "keyword":
            assert row["query_id"] not in query_text_by_id
            query_text_by_id[row["query_id"]] = row["query"]
assert len(query_text_by_id) == 30
assert len(set(query_text_by_id.values())) == 30

cached_text_hashes: set[str] = set()
cached_item_count = 0
for cache_path in CACHE_FILES:
    with np.load(cache_path, allow_pickle=False) as cached:
        hashes = cached["hashes"].tolist()
    assert all(isinstance(value, str) and re.fullmatch(r"[0-9a-f]{64}", value) for value in hashes)
    cached_item_count += len(hashes)
    cached_text_hashes.update(hashes)
assert cached_item_count == 357
cached_unique_hash_count = len(cached_text_hashes)
assert cached_unique_hash_count == 316

all_chunks: list[dict] = []
all_hierarchy: list[dict] = []
line_audits: list[dict] = []
for raw_rel, expected_sha in RAW_SOURCES:
    chunks, hierarchy, audit = parse_card(ROOT / raw_rel, expected_sha)
    all_chunks.extend(chunks)
    all_hierarchy.extend(hierarchy)
    line_audits.append(audit)

assert len({row["chunk_id"] for row in all_chunks}) == len(all_chunks)
assert len({row["node_id"] for row in all_hierarchy}) == len(all_hierarchy)
assert {row["metadata"]["card_key"] for row in all_chunks} == {
    str((ROOT / raw_rel).parent.name) + "/" + (ROOT / raw_rel).stem
    for raw_rel, _ in RAW_SOURCES
}
assert all(0 < len(row["body"]) <= MAX_BODY_CHARS for row in all_chunks)
assert all(row["retrieval_text"].strip() and row["evidence_text"].strip() for row in all_chunks)
assert all(
    row["metadata"]["issuer"] not in row["evidence_text"].splitlines()[:1]
    or row["heading_path"] and row["heading_path"][0] == row["metadata"]["issuer"]
    for row in all_chunks
)

page_spanning_rows = [
    {
        "chunk_id": row["chunk_id"],
        "card_key": row["metadata"]["card_key"],
        "node_id": row["metadata"]["node_id"],
        "part_index": row["metadata"]["part_index"],
        "page_start": row["metadata"]["page_start"],
        "page_end": row["metadata"]["page_end"],
        "page_numbers": json.dumps(row["metadata"]["page_numbers"], ensure_ascii=False),
        "body_chars": len(row["body"]),
        "heading_path": " > ".join(row["heading_path"]),
    }
    for row in all_chunks
    if row["metadata"]["page_start"] != row["metadata"]["page_end"]
]
short_body_rows = [
    {
        "chunk_id": row["chunk_id"],
        "card_key": row["metadata"]["card_key"],
        "node_id": row["metadata"]["node_id"],
        "part_index": row["metadata"]["part_index"],
        "body_chars": len(row["body"]),
        "body_nonspace_chars": len(re.sub(r"\s+", "", row["body"])),
        "heading_path": " > ".join(row["heading_path"]),
        "page_start": row["metadata"]["page_start"],
        "page_end": row["metadata"]["page_end"],
    }
    for row in all_chunks
    if len(row["body"]) < SHORT_BODY_CHARS
]
split_rows = [
    {
        "card_key": row["card_key"],
        "node_id": row["node_id"],
        "heading_path": " > ".join(row["heading_path"]),
        "direct_body_chars": row["direct_body_chars"],
        "part_count": row["part_count"],
        "search_chunk_ids": json.dumps(row["search_chunk_ids"], ensure_ascii=False),
        "page_start": row["page_start"],
        "page_end": row["page_end"],
    }
    for row in all_hierarchy
    if row["part_count"] > 1
]

print({
    "raw_files": len(RAW_SOURCES),
    "queries": len(query_text_by_id),
    "chunks": len(all_chunks),
    "hierarchy_nodes": len(all_hierarchy),
    "page_spanning_chunks": len(page_spanning_rows),
    "short_body_chunks": len(short_body_rows),
    "split_nodes": len(split_rows),
})


{'raw_files': 10, 'queries': 30, 'chunks': 147, 'hierarchy_nodes': 172, 'page_spanning_chunks': 23, 'short_body_chunks': 19, 'split_nodes': 0}


In [5]:
input_manifest = {
    "experiment_name": EXPERIMENT_NAME,
    "created_by": "Codex coder agent",
    "raw_sources": [
        {
            "path": raw_rel,
            "sha256": expected_sha,
            "issuer": (ROOT / raw_rel).parent.name,
            "card_name": (ROOT / raw_rel).stem,
            "bytes": (ROOT / raw_rel).stat().st_size,
        }
        for raw_rel, expected_sha in RAW_SOURCES
    ],
    "chunking_contract": {
        "structure_inputs": ["path issuer", "path card identifier", "page marker", "Markdown H1-H6"],
        "page_marker_regex": PAGE_RE.pattern,
        "heading_regex": HEADING_RE.pattern,
        "heading_stack_persists_across_pages": True,
        "same_or_deeper_level_replaced": True,
        "skipped_level_parent": "nearest_existing_higher_heading",
        "preamble": "root_direct_body",
        "child_body_copied_to_parent": False,
        "max_direct_body_chars": MAX_BODY_CHARS,
        "split_order": ["paragraph_boundary", "line_boundary"],
        "truncate": False,
        "overlap": False,
        "heading_only_node_searchable": False,
    },
}
write_json(OUTPUT_DIR / "input_manifest.json", input_manifest)
input_manifest_raw_sha256 = raw_sha256(OUTPUT_DIR / "input_manifest.json")

write_jsonl(OUTPUT_DIR / "chunks.jsonl", all_chunks)
write_jsonl(OUTPUT_DIR / "hierarchy_manifest.jsonl", all_hierarchy)
write_csv(
    OUTPUT_DIR / "line_assignment_audit.csv",
    line_audits,
    [
        "source_path", "source_sha256", "issuer", "card_name", "card_key",
        "raw_chars", "raw_lines", "page_marker_lines", "heading_lines",
        "direct_body_lines", "assigned_once_lines", "searchable_direct_body_lines",
        "searchable_direct_body_lines_covered_once", "hierarchy_nodes",
        "heading_only_nodes", "search_chunks",
    ],
)
write_csv(
    OUTPUT_DIR / "page_spanning_audit.csv",
    page_spanning_rows,
    [
        "chunk_id", "card_key", "node_id", "part_index", "page_start", "page_end",
        "page_numbers", "body_chars", "heading_path",
    ],
)
write_csv(
    OUTPUT_DIR / "short_body_audit.csv",
    short_body_rows,
    [
        "chunk_id", "card_key", "node_id", "part_index", "body_chars",
        "body_nonspace_chars", "heading_path", "page_start", "page_end",
    ],
)
write_csv(
    OUTPUT_DIR / "split_audit.csv",
    split_rows,
    [
        "card_key", "node_id", "heading_path", "direct_body_chars", "part_count",
        "search_chunk_ids", "page_start", "page_end",
    ],
)
write_json(OUTPUT_DIR / "evaluation_contract.json", EVALUATION_CONTRACT)

encoding = tiktoken.encoding_for_model(MODEL)
document_hashes = [text_sha256(row["retrieval_text"]) for row in all_chunks]
unique_document_text_by_hash: dict[str, str] = {}
for row, value_hash in zip(all_chunks, document_hashes):
    unique_document_text_by_hash.setdefault(value_hash, row["retrieval_text"])
query_hashes = [text_sha256(text) for text in query_text_by_id.values()]
all_unique_text_by_hash = dict(unique_document_text_by_hash)
for query_id, query_text in query_text_by_id.items():
    all_unique_text_by_hash.setdefault(text_sha256(query_text), query_text)

reused_hashes = sorted(set(all_unique_text_by_hash) & cached_text_hashes)
new_text_by_hash = {
    value_hash: text
    for value_hash, text in all_unique_text_by_hash.items()
    if value_hash not in cached_text_hashes
}
new_char_count = sum(len(text) for text in new_text_by_hash.values())
new_token_count = sum(len(encoding.encode(text)) for text in new_text_by_hash.values())
total_char_count = sum(len(text) for text in all_unique_text_by_hash.values())
total_token_count = sum(len(encoding.encode(text)) for text in all_unique_text_by_hash.values())
expected_requests = math.ceil(len(new_text_by_hash) / BATCH_SIZE)

embedding_plan = {
    "experiment_name": EXPERIMENT_NAME,
    "status": "preflight_only_awaiting_explicit_embedding_approval",
    "model": MODEL,
    "input_manifest_path": str((OUTPUT_DIR / "input_manifest.json").relative_to(ROOT)),
    "input_manifest_raw_sha256": input_manifest_raw_sha256,
    "query_count": len(query_text_by_id),
    "new_chunk_count": len(all_chunks),
    "unique_retrieval_document_count": len(unique_document_text_by_hash),
    "unique_query_text_count": len(set(query_hashes)),
    "unique_embedding_text_count": len(all_unique_text_by_hash),
    "existing_cache_file_count": len(CACHE_FILES),
    "existing_cache_item_count": cached_item_count,
    "existing_cache_unique_hash_count": cached_unique_hash_count,
    "existing_cache_reuse_unique_count": len(reused_hashes),
    "existing_cache_reuse_query_count": sum(value_hash in cached_text_hashes for value_hash in query_hashes),
    "existing_cache_reuse_document_count": sum(value_hash in cached_text_hashes for value_hash in unique_document_text_by_hash),
    "new_transmission_unique_count": len(new_text_by_hash),
    "new_transmission_character_count": new_char_count,
    "new_transmission_token_count": new_token_count,
    "all_unique_character_count": total_char_count,
    "all_unique_token_count": total_token_count,
    "batch_size": BATCH_SIZE,
    "expected_request_count": expected_requests,
    "issuer_count": len({row["metadata"]["issuer"] for row in all_chunks}),
    "card_count": len({row["metadata"]["card_key"] for row in all_chunks}),
    "issuers": sorted({row["metadata"]["issuer"] for row in all_chunks}),
    "cards": sorted({row["metadata"]["card_key"] for row in all_chunks}),
    "approval_caps": APPROVAL_CAPS,
    "within_approval_caps": {
        "items": len(new_text_by_hash) <= APPROVAL_CAPS["max_new_items"],
        "tokens": new_token_count <= APPROVAL_CAPS["max_input_tokens"],
        "requests": expected_requests <= APPROVAL_CAPS["max_requests"],
    },
    "official_unit_price_verified": False,
    "estimated_cost_usd": None,
    "cost_status": "undetermined_pending_official_pricing_confirmation",
    "network_requests_executed": 0,
    "api_requests_executed": 0,
    "new_embeddings_created": 0,
    "api_key_read": False,
}
assert all(embedding_plan["within_approval_caps"].values())
assert embedding_plan["existing_cache_reuse_query_count"] == 30
write_json(OUTPUT_DIR / "embedding_plan.json", embedding_plan)

chunk_stats = {
    "experiment_name": EXPERIMENT_NAME,
    "raw_file_count": len(RAW_SOURCES),
    "raw_character_count": sum(row["raw_chars"] for row in line_audits),
    "raw_line_count": sum(row["raw_lines"] for row in line_audits),
    "page_marker_line_count": sum(row["page_marker_lines"] for row in line_audits),
    "heading_line_count": sum(row["heading_lines"] for row in line_audits),
    "direct_body_line_count": sum(row["direct_body_lines"] for row in line_audits),
    "hierarchy_node_count": len(all_hierarchy),
    "heading_only_node_count": sum(row["heading_only"] for row in all_hierarchy),
    "search_chunk_count": len(all_chunks),
    "page_spanning_chunk_count": len(page_spanning_rows),
    "short_body_threshold_chars": SHORT_BODY_CHARS,
    "short_body_chunk_count": len(short_body_rows),
    "split_node_count": len(split_rows),
    "body_chars": {
        "min": min(len(row["body"]) for row in all_chunks),
        "median": statistics.median(len(row["body"]) for row in all_chunks),
        "p95": float(np.percentile([len(row["body"]) for row in all_chunks], 95)),
        "max": max(len(row["body"]) for row in all_chunks),
    },
}
write_json(OUTPUT_DIR / "chunk_stats.json", chunk_stats)
print(json.dumps(embedding_plan, ensure_ascii=False, indent=2))


{
  "experiment_name": "구조 기반 청킹 + 제목 경로 검색문",
  "status": "preflight_only_awaiting_explicit_embedding_approval",
  "model": "text-embedding-3-small",
  "input_manifest_path": "notebooks/data/22_structural_heading_chunking_ablation/input_manifest.json",
  "input_manifest_raw_sha256": "7c6756b14e6395566b93240b3003aeac1125fabcec43a1cfd987102bb8e7d564",
  "query_count": 30,
  "new_chunk_count": 147,
  "unique_retrieval_document_count": 147,
  "unique_query_text_count": 30,
  "unique_embedding_text_count": 177,
  "existing_cache_file_count": 6,
  "existing_cache_item_count": 357,
  "existing_cache_unique_hash_count": 316,
  "existing_cache_reuse_unique_count": 30,
  "existing_cache_reuse_query_count": 30,
  "existing_cache_reuse_document_count": 0,
  "new_transmission_unique_count": 147,
  "new_transmission_character_count": 74077,
  "new_transmission_token_count": 65856,
  "all_unique_character_count": 74842,
  "all_unique_token_count": 66609,
  "batch_size": 64,
  "expected_request_count

## 명시적 실행 중단 경계

아래 검증 셀은 preflight 산출물과 원본 불변성만 확인합니다. 이 노트북에는 API client 생성, API key 조회, embedding 요청, 네트워크 호출, Chroma client/query 셀이 없습니다. 신규 embedding은 embedding_plan.json의 정확한 수치와 승인 상한을 검토해 별도 승인을 받은 뒤에만 후속 단계로 추가할 수 있습니다.

In [6]:
readme = f"""# 22. 구조 기반 청킹 + 제목 경로 검색문

이 폴더는 개발 질의 30개를 위한 오프라인 preflight 결과입니다. raw TXT의 page marker와 Markdown heading만으로 direct body 청크를 만들고, 검색용 문장에는 issuer/card/heading path를 붙였습니다. 근거 확인용 evidence_text에는 인위적으로 붙인 issuer/card를 넣지 않았습니다.

## 이번 실행에서 한 일

- 구조 청크 {len(all_chunks):,}개와 hierarchy node {len(all_hierarchy):,}개를 생성했습니다.
- raw body line을 direct body에 한 번씩만 배정했는지 검사했습니다.
- page-spanning, {SHORT_BODY_CHARS}자 미만 body, 4,000자 분할을 각각 CSV로 남겼습니다.
- 기존 cache hash와 비교해 재사용/신규 전송 계획을 계산했습니다.
- API key, 네트워크, embedding API, Chroma query는 사용하지 않았습니다.

## 파일

- chunks.jsonl: body, heading_path, retrieval_text, evidence_text와 metadata
- hierarchy_manifest.jsonl: 검색에서 제외되는 heading-only parent까지 포함한 구조
- line_assignment_audit.csv: 원문 line 일대일 배정 감사
- page_spanning_audit.csv, short_body_audit.csv, split_audit.csv: 경계 위험 감사
- input_manifest.json, chunk_stats.json: 입력과 청크 통계
- embedding_plan.json: 캐시 재사용, 신규 전송량, 토큰/요청 상한과 비용 미확정 상태
- evaluation_contract.json: 향후 ranking freeze와 개발셋 평가 규칙
- integrity.json: 입력 불변성과 산출물 hash

## 승인 경계와 한계

현재 결과는 single-run, development-only preflight입니다. 공식 단가를 확인하지 않았으므로 비용은 미확정입니다. embedding_plan의 신규 전송 수와 토큰 수를 검토해 명시적으로 승인하기 전에는 API를 실행하면 안 됩니다. 아직 vector/BM25/RRF ranking과 relevance 평가를 하지 않았으며, 운영 또는 별도 일반화 평가에 관한 결론도 없습니다.

향후 Evidence relevance는 level을 무시하고 expected card와 required terms가 evidence_text에 있는지를 봅니다. 제목 경로만으로 required term이 충족되는 부풀림 가능성이 있어 heading-only match를 반드시 별도 감사해야 합니다. Recall과 nDCG는 새 청크 수에 따라 분모가 달라지므로 진단 지표로만 사용합니다.
"""
(OUTPUT_DIR / "README.md").write_text(readme, encoding="utf-8")

SOURCE_HASHES_AFTER = source_hash_map()
assert SOURCE_HASHES_AFTER == SOURCE_HASHES_BEFORE

expected_output_files = [
    "README.md",
    "chunk_stats.json",
    "chunks.jsonl",
    "embedding_plan.json",
    "evaluation_contract.json",
    "hierarchy_manifest.jsonl",
    "input_manifest.json",
    "line_assignment_audit.csv",
    "page_spanning_audit.csv",
    "short_body_audit.csv",
    "split_audit.csv",
]
output_hashes = {
    name: raw_sha256(OUTPUT_DIR / name)
    for name in expected_output_files
}
assert len(output_hashes) == len(expected_output_files)
assert all(re.fullmatch(r"[0-9a-f]{64}", value) for value in output_hashes.values())

with (OUTPUT_DIR / "chunks.jsonl").open(encoding="utf-8") as handle:
    saved_chunks = [json.loads(line) for line in handle if line.strip()]
with (OUTPUT_DIR / "hierarchy_manifest.jsonl").open(encoding="utf-8") as handle:
    saved_hierarchy = [json.loads(line) for line in handle if line.strip()]
assert saved_chunks == all_chunks
assert saved_hierarchy == all_hierarchy

integrity = {
    "status": "PASS",
    "experiment_name": EXPERIMENT_NAME,
    "fresh_kernel_preflight": True,
    "source_hashes_before": SOURCE_HASHES_BEFORE,
    "source_hashes_after": SOURCE_HASHES_AFTER,
    "source_unchanged": SOURCE_HASHES_BEFORE == SOURCE_HASHES_AFTER,
    "raw_file_count": len(RAW_SOURCES),
    "query_count": len(query_text_by_id),
    "existing_chunk_count_contract": 327,
    "existing_cache_file_count": len(CACHE_FILES),
    "existing_cache_item_count": cached_item_count,
    "existing_cache_unique_hash_count": cached_unique_hash_count,
    "chroma_file_count": len(CHROMA_FILES),
    "new_chunk_count": len(all_chunks),
    "hierarchy_node_count": len(all_hierarchy),
    "chunk_id_unique": len({row["chunk_id"] for row in all_chunks}) == len(all_chunks),
    "node_id_unique": len({row["node_id"] for row in all_hierarchy}) == len(all_hierarchy),
    "all_body_chars_within_limit": all(0 < len(row["body"]) <= MAX_BODY_CHARS for row in all_chunks),
    "line_assignment_pass": all(
        row["raw_lines"] == row["assigned_once_lines"]
        and row["searchable_direct_body_lines"] == row["searchable_direct_body_lines_covered_once"]
        for row in line_audits
    ),
    "heading_only_nodes_not_searchable": all(
        not row["search_chunk_ids"] for row in all_hierarchy if row["heading_only"]
    ),
    "network_requests": 0,
    "api_requests": 0,
    "new_embeddings": 0,
    "api_key_read": False,
    "chroma_client_opened": False,
    "chroma_queries": 0,
    "output_raw_sha256_excluding_integrity_and_notebook": output_hashes,
    "self_hash_policy": "integrity.json and executed notebook excluded from in-kernel output hash map",
}
write_json(OUTPUT_DIR / "integrity.json", integrity)

saved_integrity = json.loads((OUTPUT_DIR / "integrity.json").read_text(encoding="utf-8"))
assert saved_integrity["status"] == "PASS"
assert saved_integrity["source_unchanged"]
assert saved_integrity["network_requests"] == 0
assert saved_integrity["api_requests"] == 0
assert saved_integrity["new_embeddings"] == 0
assert source_hash_map() == SOURCE_HASHES_BEFORE

print(json.dumps({
    "integrity": integrity["status"],
    "chunks": len(all_chunks),
    "nodes": len(all_hierarchy),
    "new_transmission_count": embedding_plan["new_transmission_unique_count"],
    "new_tokens": embedding_plan["new_transmission_token_count"],
    "expected_requests_after_approval": embedding_plan["expected_request_count"],
    "source_unchanged": integrity["source_unchanged"],
    "network_api_embedding": [0, 0, 0],
}, ensure_ascii=False, indent=2))


{
  "integrity": "PASS",
  "chunks": 147,
  "nodes": 172,
  "new_transmission_count": 147,
  "new_tokens": 65856,
  "expected_requests_after_approval": 3,
  "source_unchanged": true,
  "network_api_embedding": [
    0,
    0,
    0
  ]
}


## 승인 후 단계 — embedding, ranking freeze, 개발셋 평가

승인 범위는 신규 retrieval_text 147개, 최대 65,856 tokens와 3 requests입니다. 기존 query embedding 30개는 13번 cache에서 읽기 전용으로 재사용합니다. exact duplicate는 같은 card_key의 NFKC/lower/공백 정규화 body equality excess, containment는 exact duplicate를 제외하고 같은 card의 RAW_TOKEN 고유 집합 중 짧은 쪽 최소 5 tokens, 교집합 비율 0.8 이상 pair 수로 결과 전에 고정합니다.

In [7]:
from collections import defaultdict
from decimal import Decimal
import tempfile

PERSISTED_PLAN=json.loads((OUTPUT_DIR/"embedding_plan.json").read_text(encoding="utf-8"))
assert PERSISTED_PLAN==embedding_plan
assert PERSISTED_PLAN["input_manifest_raw_sha256"]=="7c6756b14e6395566b93240b3003aeac1125fabcec43a1cfd987102bb8e7d564"
assert raw_sha256(OUTPUT_DIR/"input_manifest.json")==PERSISTED_PLAN["input_manifest_raw_sha256"]
assert tuple(PERSISTED_PLAN[key] for key in ("new_chunk_count","unique_retrieval_document_count","new_transmission_unique_count","new_transmission_token_count","expected_request_count"))==(147,147,147,65856,3)
EMBEDDING_APPROVAL_CAP={"items":147,"tokens":65856,"requests":3}
REDUNDANCY_CONTRACT={"scope":"same_card_only","body_normalization":"NFKC + lower + collapse whitespace","exact_duplicate":"normalized body equality; report excess count","containment_token_representation":"RAW_TOKEN unique set","minimum_shorter_body_unique_tokens":5,"containment_threshold":0.8,"exact_duplicates_excluded_from_containment":True}
FULL_EVALUATION_CONTRACT={**EVALUATION_CONTRACT,"approved_embedding":{"items":147,"planned_tokens":65856,"max_requests":3,"model":MODEL,"pricing_usd_per_million_input_tokens":0.02,"pricing_source_url":"https://openai.com/api/pricing/","pricing_basis":"user-approved current official price contract"},"redundancy_definition":REDUNDANCY_CONTRACT}
write_json(OUTPUT_DIR/"evaluation_contract.json",FULL_EVALUATION_CONTRACT)
encoder=tiktoken.encoding_for_model(MODEL)
approved_embedding_items=[(row["chunk_id"],row["retrieval_text"]) for row in all_chunks]
approved_hashes=[text_sha256(text) for _,text in approved_embedding_items]
approved_tokens=[len(encoder.encode(text)) for _,text in approved_embedding_items]
assert len(approved_embedding_items)==147==len(set(approved_hashes))
assert sum(approved_tokens)==65856 and math.ceil(len(approved_embedding_items)/BATCH_SIZE)==3
assert set(approved_hashes)==set(new_text_by_hash)
assert all(new_text_by_hash[value_hash]==text for (_,text),value_hash in zip(approved_embedding_items,approved_hashes))
embedding_target_manifest={"model":MODEL,"input_manifest_raw_sha256":PERSISTED_PLAN["input_manifest_raw_sha256"],"embedding_plan_raw_sha256":raw_sha256(OUTPUT_DIR/"embedding_plan.json"),"count":147,"planned_tokens":65856,"batch_size":BATCH_SIZE,"planned_requests":3,"items":[{"order":index,"chunk_id":chunk_id,"text_sha256":value_hash,"characters":len(text),"tokens":tokens,"batch":(index-1)//BATCH_SIZE+1} for index,((chunk_id,text),value_hash,tokens) in enumerate(zip(approved_embedding_items,approved_hashes,approved_tokens),start=1)]}
write_json(OUTPUT_DIR/"embedding_target_manifest.json",embedding_target_manifest)
EMBEDDING_TARGET_MANIFEST_SHA256=raw_sha256(OUTPUT_DIR/"embedding_target_manifest.json")
print({"items":147,"tokens":65856,"requests":3,"target_manifest_sha256":EMBEDDING_TARGET_MANIFEST_SHA256})


{'items': 147, 'tokens': 65856, 'requests': 3, 'target_manifest_sha256': '72e45029ffb1907da0e4b4a7bb3b7fbc54de947474c556386100b56ceac59b75'}


In [8]:
CACHE_22_ROOT=OUTPUT_DIR/"embedding_cache"/MODEL
CACHE_22_ROOT.mkdir(parents=True,exist_ok=True)
approved_live_api=os.getenv("RUN_APPROVED_22_EMBEDDING","0")=="1"
api_client=None
current_run_api_requests=current_run_api_input_tokens=0
batch_usage=[]
new_vectors_by_chunk_id={}
for batch_start in range(0,len(approved_embedding_items),BATCH_SIZE):
    batch_number=batch_start//BATCH_SIZE+1
    batch=approved_embedding_items[batch_start:batch_start+BATCH_SIZE]
    chunk_ids_batch,texts=zip(*batch)
    text_hashes=[text_sha256(text) for text in texts]
    fingerprint=text_sha256(canonical_json({"model":MODEL,"hashes":text_hashes}))
    cache_path=CACHE_22_ROOT/(fingerprint+".npz")
    if cache_path.is_file():
        with np.load(cache_path,allow_pickle=False) as cached:
            vectors=cached["embeddings"].copy()
            assert cached["hashes"].tolist()==text_hashes
            creation_tokens=int(cached["creation_input_tokens"].item())
            creation_requests=int(cached["creation_api_requests"].item())
        cache_hit=True
    else:
        assert approved_live_api and current_run_api_requests<3
        from dotenv import get_key
        from openai import OpenAI
        api_key=get_key(str(ROOT/".env"),"OPENAI_API_KEY")
        if not api_key:
            raise RuntimeError("OPENAI_API_KEY unavailable; value not printed or persisted")
        api_client=api_client or OpenAI(api_key=api_key,max_retries=0,timeout=120.0)
        try:
            response=api_client.embeddings.create(model=MODEL,input=list(texts),encoding_format="float")
        except Exception as error:
            write_json(OUTPUT_DIR/"embedding_failure.json",{"batch":batch_number,"error_type":type(error).__name__,"secret_persisted":False})
            raise
        finally:
            del api_key
        ordered=sorted(response.data,key=lambda item:item.index)
        assert [item.index for item in ordered]==list(range(len(batch)))
        vectors=np.asarray([item.embedding for item in ordered],dtype=np.float32)
        creation_tokens=int(response.usage.prompt_tokens)
        creation_requests=1
        current_run_api_requests+=1
        current_run_api_input_tokens+=creation_tokens
        descriptor,temporary_name=tempfile.mkstemp(dir=CACHE_22_ROOT,suffix=".npz")
        os.close(descriptor)
        np.savez_compressed(temporary_name,embeddings=vectors,hashes=np.asarray(text_hashes),creation_input_tokens=np.asarray(creation_tokens,dtype=np.int64),creation_api_requests=np.asarray(creation_requests,dtype=np.int64))
        os.replace(temporary_name,cache_path)
        cache_hit=False
    assert vectors.shape==(len(batch),1536) and vectors.dtype==np.float32 and np.isfinite(vectors).all()
    new_vectors_by_chunk_id.update(dict(zip(chunk_ids_batch,vectors)))
    batch_usage.append({"batch":batch_number,"items":len(batch),"cached_this_run":cache_hit,"batch_fingerprint":fingerprint,"cache_path":str(cache_path.relative_to(ROOT)),"cache_raw_sha256":raw_sha256(cache_path),"creation_api_requests":creation_requests,"creation_input_tokens":creation_tokens,"dimension":1536,"dtype":"float32","finite":True})
assert len(new_vectors_by_chunk_id)==147
creation_api_requests_total=sum(row["creation_api_requests"] for row in batch_usage)
creation_api_input_tokens_total=sum(row["creation_input_tokens"] for row in batch_usage)
assert creation_api_requests_total==3 and creation_api_input_tokens_total<=65856
actual_cost_usd=creation_api_input_tokens_total*0.02/1_000_000
(OUTPUT_DIR/"embedding_failure.json").unlink(missing_ok=True)
embedding_usage_22={"model":MODEL,"target_manifest_raw_sha256":EMBEDDING_TARGET_MANIFEST_SHA256,"items":147,"dimension":1536,"dtype":"float32","finite":True,"batch_size":BATCH_SIZE,"batch_count":len(batch_usage),"batches":batch_usage,"creation_api_requests_total":creation_api_requests_total,"creation_api_input_tokens_total":creation_api_input_tokens_total,"creation_cost_usd":actual_cost_usd,"pricing_usd_per_million_input_tokens":0.02,"pricing_source_url":"https://openai.com/api/pricing/","current_run_api_requests":current_run_api_requests,"current_run_api_input_tokens":current_run_api_input_tokens,"current_run_cache_hit_batches":sum(row["cached_this_run"] for row in batch_usage),"other_network_requests":0,"secret_persisted":False}
write_json(OUTPUT_DIR/"embedding_usage.json",embedding_usage_22)
print({key:embedding_usage_22[key] for key in ("creation_api_requests_total","creation_api_input_tokens_total","creation_cost_usd","current_run_api_requests","current_run_cache_hit_batches")})


{'creation_api_requests_total': 3, 'creation_api_input_tokens_total': 65856, 'creation_cost_usd': 0.0013171200000000002, 'current_run_api_requests': 0, 'current_run_cache_hit_batches': 3}


## Gold를 읽기 전 ranking freeze

query text, corpus search text와 승인 vector만으로 기존 327개와 신규 147개의 RRF Top50을 먼저 저장·hash합니다. 이 셀은 평가 label을 읽지 않습니다.

In [9]:
RAW_TOKEN=re.compile(r"[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*",re.IGNORECASE)
BM25_K1,BM25_B=1.5,0.75
RRF_VECTOR_WEIGHT,RRF_BM25_WEIGHT,RRF_K,RRF_DEPTH=0.4,0.6,60,50
normalized_text=lambda value:" ".join(unicodedata.normalize("NFKC",str(value)).lower().split())
def canonical_decimal(value):
    rendered=format(Decimal(str(value).replace(",","")).normalize(),"f")
    rendered=rendered.rstrip("0").rstrip(".") if "." in rendered else rendered
    return "0" if rendered in {"","-0"} else rendered
def search_tokens(value):
    text=normalized_text(value); tokens=list(RAW_TOKEN.findall(text))
    for run in re.findall(r"[가-힣](?:[가-힣 ]{0,38}[가-힣])?",text):
        joined=run.replace(" ","")
        for size in (2,3,4):
            tokens.extend("ko"+str(size)+"_"+joined[index:index+size] for index in range(max(0,len(joined)-size+1)))
    consumed=[]
    for match in re.finditer(r"(\d[\d,]*(?:\.\d+)?)\s*만\s*(\d[\d,]*(?:\.\d+)?)\s*천\s*원",text):
        amount=Decimal(match.group(1).replace(",",""))*10000+Decimal(match.group(2).replace(",",""))*1000
        tokens.append("money_krw_"+canonical_decimal(amount)); consumed.append(match.span())
    for match in re.finditer(r"(\d[\d,]*(?:\.\d+)?)\s*(만|천)?\s*원",text):
        if any(left<=match.start() and match.end()<=right for left,right in consumed): continue
        multiplier={"만":10000,"천":1000,None:1}[match.group(2)]
        tokens.append("money_krw_"+canonical_decimal(Decimal(match.group(1).replace(",",""))*multiplier))
    for match in re.finditer(r"(\d[\d,]*(?:\.\d+)?)\s*%",text):
        tokens.append("percent_"+canonical_decimal(match.group(1)))
    for match in re.finditer(r"(?:(월|연|년|일)\s*)?(\d[\d,]*(?:\.\d+)?)\s*(회|개월|년|일)",text):
        tokens.append("period_"+(match.group(1) or "none")+"_"+canonical_decimal(match.group(2))+"_"+match.group(3))
    return tokens
def bm25_scores(query_tokens,documents):
    tokenized={identifier:list(tokens) for identifier,tokens in documents.items()}
    document_frequency=Counter(token for tokens in tokenized.values() for token in set(tokens))
    average_length=sum(map(len,tokenized.values()))/len(tokenized)
    scores={}
    for identifier,tokens in tokenized.items():
        frequencies,score=Counter(tokens),0.0
        for token in query_tokens:
            frequency=frequencies[token]
            if frequency:
                inverse_frequency=math.log(1+(len(tokenized)-document_frequency[token]+0.5)/(document_frequency[token]+0.5))
                score+=inverse_frequency*frequency*(BM25_K1+1)/(frequency+BM25_K1*(1-BM25_B+BM25_B*len(tokens)/average_length))
        scores[identifier]=score
    return scores
def rank_scores(scores):
    return [identifier for identifier,_ in sorted(scores.items(),key=lambda item:(-item[1],item[0]))]
def l2_rank(ids,matrix,query_vector):
    distances=np.sum((matrix-query_vector)**2,axis=1)
    return [identifier for identifier,_ in sorted(zip(ids,distances.tolist()),key=lambda item:(item[1],item[0]))]
def weighted_rrf(vector_ranking,bm25_ranking):
    scores=defaultdict(float)
    for weight,ranking in ((RRF_VECTOR_WEIGHT,vector_ranking[:RRF_DEPTH]),(RRF_BM25_WEIGHT,bm25_ranking[:RRF_DEPTH])):
        for rank,identifier in enumerate(ranking,start=1): scores[identifier]+=weight/(RRF_K+rank)
    ranking=[identifier for identifier,_ in sorted(scores.items(),key=lambda item:(-item[1],item[0]))]
    return ranking,scores

existing_chunks=[json.loads(line) for line in SOURCE_CHUNKS.read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(existing_chunks)==327
existing_items=[("chunk:"+row["id"],row["document"]) for row in existing_chunks]+[("query:"+query_id,text) for query_id,text in query_text_by_id.items()]
existing_vectors={}
for batch_index,batch_start in enumerate(range(0,len(existing_items),BATCH_SIZE)):
    batch=existing_items[batch_start:batch_start+BATCH_SIZE]
    hashes=[text_sha256(text) for _,text in batch]
    fingerprint=text_sha256(canonical_json({"model":MODEL,"hashes":hashes}))
    assert fingerprint==CACHE_FINGERPRINTS[batch_index]
    with np.load(CACHE_FILES[batch_index],allow_pickle=False) as cached:
        vectors=cached["embeddings"]
        assert cached["hashes"].tolist()==hashes and vectors.shape==(len(batch),1536)
        assert vectors.dtype==np.float32 and np.isfinite(vectors).all()
        existing_vectors.update({key:vector.copy() for (key,_),vector in zip(batch,vectors)})
assert len(existing_vectors)==357
query_vectors={query_id:existing_vectors["query:"+query_id] for query_id in query_text_by_id}
existing_ids=[row["id"] for row in existing_chunks]; new_ids=[row["chunk_id"] for row in all_chunks]
existing_matrix=np.stack([existing_vectors["chunk:"+identifier] for identifier in existing_ids])
new_matrix=np.stack([new_vectors_by_chunk_id[identifier] for identifier in new_ids])
assert existing_matrix.shape==(327,1536) and new_matrix.shape==(147,1536)
corpora={
 "existing_exact_v1":{"ids":existing_ids,"texts":{row["id"]:row["document"] for row in existing_chunks},"matrix":existing_matrix},
 "structural_heading_path":{"ids":new_ids,"texts":{row["chunk_id"]:row["retrieval_text"] for row in all_chunks},"matrix":new_matrix}}
rankings={}; ranking_freeze_rows=[]
for configuration,corpus in corpora.items():
    tokenized={identifier:search_tokens(text) for identifier,text in corpus["texts"].items()}
    for query_id,query_text in query_text_by_id.items():
        bm25=rank_scores(bm25_scores(search_tokens(query_text),tokenized))
        vector=l2_rank(corpus["ids"],corpus["matrix"],query_vectors[query_id])
        fused,scores=weighted_rrf(vector,bm25); top50=fused[:50]
        assert len(top50)==len(set(top50))==50
        assert top50==sorted(top50,key=lambda identifier:(-scores[identifier],identifier))
        rankings[(configuration,query_id)]={"top50":top50,"vector":vector,"bm25":bm25,"scores":scores}
        ranking_freeze_rows.append({"configuration":configuration,"query_id":query_id,
          "top50_chunk_ids":top50,"top50_total_scores":[scores[item] for item in top50],
          "vector_top50_chunk_ids":vector[:50],"bm25_top50_chunk_ids":bm25[:50],
          "component_depth":50,"rrf_k":60,"vector_weight":0.4,"bm25_weight":0.6})
assert len(ranking_freeze_rows)==60
write_jsonl(OUTPUT_DIR/"ranking_freeze.jsonl",ranking_freeze_rows)
ranking_freeze_sha256=raw_sha256(OUTPUT_DIR/"ranking_freeze.jsonl")
ranking_freeze_manifest={"gold_loaded_before_freeze":False,"row_count":60,"query_count":30,
 "configurations":list(corpora),"depth":50,"ranking_freeze_raw_sha256":ranking_freeze_sha256,
 "input_manifest_raw_sha256":PERSISTED_PLAN["input_manifest_raw_sha256"],
 "embedding_target_manifest_raw_sha256":EMBEDDING_TARGET_MANIFEST_SHA256}
write_json(OUTPUT_DIR/"ranking_freeze.json",ranking_freeze_manifest)
assert raw_sha256(OUTPUT_DIR/"ranking_freeze.jsonl")==ranking_freeze_sha256
print({"ranking_rows":60,"ranking_freeze_sha256":ranking_freeze_sha256,"gold_loaded_before_freeze":False})


{'ranking_rows': 60, 'ranking_freeze_sha256': 'e7d6b5db7fe9852f131e597d2b2aa198a9125f34fdac2674e3fa0f3358192734', 'gold_loaded_before_freeze': False}


## Ranking freeze 이후 gold 평가

고정된 Top50을 바꾸지 않고 expected_card, required_terms, category를 평가와 grouping에만 사용합니다. Evidence relevance는 level을 무시하며 신규 corpus는 evidence_text만 검사합니다.

In [10]:
assert raw_sha256(OUTPUT_DIR/"ranking_freeze.jsonl")==ranking_freeze_sha256
all_baseline_rows=list(csv.DictReader(QUERY_CSV.open(encoding="utf-8",newline="")))
gold_by_id={row["query_id"]:{"query_id":row["query_id"],"category":row["category"],"expected_card":row["expected_card"],"expected_level":row["expected_level"],"required_terms":json.loads(row["required_terms"])} for row in all_baseline_rows if row["method"]=="keyword"}
assert set(gold_by_id)==set(query_text_by_id) and len(gold_by_id)==30
assert sum(row["expected_level"]=="card" for row in gold_by_id.values())==10
assert Counter(row["category"] for row in gold_by_id.values())=={"proper_noun":10,"numeric_condition":10,"semantic":10}
existing_by_id={row["id"]:row for row in existing_chunks}; new_by_id={row["chunk_id"]:row for row in all_chunks}
chunk_maps={"existing_exact_v1":existing_by_id,"structural_heading_path":new_by_id}
def card_key_for(configuration,identifier): return chunk_maps[configuration][identifier]["metadata"]["card_key"]
def evidence_for(configuration,identifier):
    row=chunk_maps[configuration][identifier]
    return row["document"] if configuration=="existing_exact_v1" else row["evidence_text"]
def answer_relevant_ids(configuration,gold):
    return {identifier for identifier in chunk_maps[configuration] if card_key_for(configuration,identifier)==gold["expected_card"] and all(normalized_text(term) in normalized_text(evidence_for(configuration,identifier)) for term in gold["required_terms"])}
def evaluate_query(configuration,gold,ranking):
    card_hits=[card_key_for(configuration,item)==gold["expected_card"] for item in ranking[:5]]
    card_first=next((rank for rank,hit in enumerate(card_hits,start=1) if hit),None)
    result={"card_hit_at_3":int(any(card_hits[:3])),"card_mrr_at_5":1/card_first if card_first else 0.0}
    if gold["expected_level"]=="card":
        return {**result,"answer_hit_at_3":None,"answer_recall_at_5":None,"answer_mrr_at_5":None,"answer_ndcg_at_5":None,"answer_relevant_count":None}
    relevant=answer_relevant_ids(configuration,gold); assert relevant,(configuration,gold["query_id"])
    hits=[item in relevant for item in ranking[:5]]
    first=next((rank for rank,hit in enumerate(hits,start=1) if hit),None)
    dcg=sum(hit/math.log2(rank+1) for rank,hit in enumerate(hits,start=1))
    ideal=sum(1/math.log2(rank+1) for rank in range(1,min(5,len(relevant))+1))
    return {**result,"answer_hit_at_3":int(any(hits[:3])),"answer_recall_at_5":sum(hits)/len(relevant),"answer_mrr_at_5":1/first if first else 0.0,"answer_ndcg_at_5":dcg/ideal,"answer_relevant_count":len(relevant)}

per_query_rows=[]; relevance_rows=[]; heading_audit_rows=[]
for configuration in corpora:
    for query_id,gold in gold_by_id.items():
        ranking=rankings[(configuration,query_id)]["top50"]
        metrics=evaluate_query(configuration,gold,ranking)
        per_query_rows.append({"configuration":configuration,"query_id":query_id,"question_group":"card" if gold["expected_level"]=="card" else "evidence","category":gold["category"],**metrics,"top5_chunk_ids":canonical_json(ranking[:5]),"top5_cards":canonical_json([card_key_for(configuration,item) for item in ranking[:5]])})
        if gold["expected_level"]!="card":
            relevant=sorted(answer_relevant_ids(configuration,gold))
            relevance_rows.append({"configuration":configuration,"query_id":query_id,"category":gold["category"],"expected_card":gold["expected_card"],"required_terms":gold["required_terms"],"relevant_chunk_ids":relevant,"relevant_count":len(relevant),"definition":"level_relaxed_term_bearing"})
            if configuration=="structural_heading_path":
                for identifier in relevant:
                    row=new_by_id[identifier]; body=normalized_text(row["body"]); heading=normalized_text(" > ".join(row["heading_path"]))
                    contributed=[term for term in gold["required_terms"] if normalized_text(term) not in body and normalized_text(term) in heading]
                    heading_audit_rows.append({"query_id":query_id,"category":gold["category"],"chunk_id":identifier,"card_key":row["metadata"]["card_key"],"heading_path":" > ".join(row["heading_path"]),"required_terms":canonical_json(gold["required_terms"]),"heading_contributed_terms":canonical_json(contributed),"heading_only_contribution":int(bool(contributed)),"body_preview":body[:240]})
assert len(per_query_rows)==60 and len(relevance_rows)==40
write_csv(OUTPUT_DIR/"retrieval_per_query.csv",per_query_rows,list(per_query_rows[0]))
write_jsonl(OUTPUT_DIR/"relevance_sets.jsonl",relevance_rows)
write_csv(OUTPUT_DIR/"heading_inflation_audit.csv",heading_audit_rows,["query_id","category","chunk_id","card_key","heading_path","required_terms","heading_contributed_terms","heading_only_contribution","body_preview"])
def aggregate(rows):
    result={}
    for metric in ("card_hit_at_3","card_mrr_at_5","answer_hit_at_3","answer_recall_at_5","answer_mrr_at_5","answer_ndcg_at_5"):
        values=[row[metric] for row in rows if row[metric] is not None]
        result[metric]=sum(values)/len(values) if values else None; result[metric+"_denominator"]=len(values)
    return result
summary_rows=[]
for configuration in corpora:
    rows=[row for row in per_query_rows if row["configuration"]==configuration]
    groups={"all":rows,"card":[row for row in rows if row["question_group"]=="card"],"evidence":[row for row in rows if row["question_group"]=="evidence"],"numeric":[row for row in rows if row["category"]=="numeric_condition"],"semantic":[row for row in rows if row["category"]=="semantic"],"proper_noun":[row for row in rows if row["category"]=="proper_noun"]}
    summary_rows.extend({"configuration":configuration,"group":group,"denominator":len(selected),**aggregate(selected)} for group,selected in groups.items())
assert len(summary_rows)==12
write_csv(OUTPUT_DIR/"retrieval_summary.csv",summary_rows,list(summary_rows[0]))
def paired_values(metric,group):
    old={row["query_id"]:row for row in per_query_rows if row["configuration"]=="existing_exact_v1"}
    new={row["query_id"]:row for row in per_query_rows if row["configuration"]=="structural_heading_path"}
    eligible=[qid for qid,gold in gold_by_id.items() if group=="all" or (group=="card" and gold["expected_level"]=="card") or (group=="evidence" and gold["expected_level"]!="card") or (group=="numeric" and gold["category"]=="numeric_condition") or (group=="semantic" and gold["category"]=="semantic")]
    return [(qid,old[qid][metric],new[qid][metric]) for qid in eligible if old[qid][metric] is not None and new[qid][metric] is not None]
paired_delta_rows=[]; paired_wlt_rows=[]
for group in ("all","card","evidence","numeric","semantic"):
    for metric in ("card_hit_at_3","card_mrr_at_5","answer_hit_at_3","answer_recall_at_5","answer_mrr_at_5","answer_ndcg_at_5"):
        values=paired_values(metric,group)
        if not values: continue
        wins=sum(new>old+1e-12 for _,old,new in values); losses=sum(new<old-1e-12 for _,old,new in values); ties=len(values)-wins-losses
        paired_delta_rows.extend({"group":group,"metric":metric,"query_id":qid,"baseline":old,"structural":new,"delta":new-old} for qid,old,new in values)
        paired_wlt_rows.append({"group":group,"metric":metric,"denominator":len(values),"wins":wins,"losses":losses,"ties":ties,"mean_delta":sum(new-old for _,old,new in values)/len(values)})
write_csv(OUTPUT_DIR/"paired_deltas.csv",paired_delta_rows,list(paired_delta_rows[0]))
write_csv(OUTPUT_DIR/"paired_wlt.csv",paired_wlt_rows,list(paired_wlt_rows[0]))
changed_rows=[]
for query_id,gold in gold_by_id.items():
    old_top5=rankings[("existing_exact_v1",query_id)]["top50"][:5]; new_top5=rankings[("structural_heading_path",query_id)]["top50"][:5]
    if old_top5!=new_top5:
        old=evaluate_query("existing_exact_v1",gold,old_top5); new=evaluate_query("structural_heading_path",gold,new_top5)
        changed_rows.append({"query_id":query_id,"category":gold["category"],"question_group":"card" if gold["expected_level"]=="card" else "evidence","old_top5_chunk_ids":canonical_json(old_top5),"new_top5_chunk_ids":canonical_json(new_top5),"old_top5_cards":canonical_json([card_key_for("existing_exact_v1",item) for item in old_top5]),"new_top5_cards":canonical_json([card_key_for("structural_heading_path",item) for item in new_top5]),"card_hit_delta":new["card_hit_at_3"]-old["card_hit_at_3"],"answer_hit_delta":None if old["answer_hit_at_3"] is None else new["answer_hit_at_3"]-old["answer_hit_at_3"],"answer_mrr_delta":None if old["answer_mrr_at_5"] is None else new["answer_mrr_at_5"]-old["answer_mrr_at_5"]})
write_csv(OUTPUT_DIR/"changed_top5.csv",changed_rows,list(changed_rows[0]))
def redundancy(configuration):
    rows=chunk_maps[configuration]
    body={identifier:normalized_text(row["document"] if configuration=="existing_exact_v1" else row["body"]) for identifier,row in rows.items()}
    groups=defaultdict(list)
    for identifier,text in body.items(): groups[(card_key_for(configuration,identifier),text)].append(identifier)
    exact_excess=sum(len(ids)-1 for ids in groups.values() if len(ids)>1)
    tokens={identifier:set(RAW_TOKEN.findall(text)) for identifier,text in body.items()}
    identifiers=list(rows); pairs=[]
    for index,left in enumerate(identifiers):
        for right in identifiers[index+1:]:
            if card_key_for(configuration,left)!=card_key_for(configuration,right) or body[left]==body[right]: continue
            shorter=min(len(tokens[left]),len(tokens[right]))
            if shorter<5: continue
            ratio=len(tokens[left]&tokens[right])/shorter
            if ratio>=0.8:
                pairs.append({"configuration":configuration,"card_key":card_key_for(configuration,left),"left_chunk_id":left,"right_chunk_id":right,"left_unique_tokens":len(tokens[left]),"right_unique_tokens":len(tokens[right]),"containment_ratio":ratio})
    return {"configuration":configuration,"chunk_count":len(rows),"exact_duplicate_group_count":sum(len(ids)>1 for ids in groups.values()),"exact_duplicate_excess":exact_excess,"containment_pair_count":len(pairs)},pairs
redundancy_rows=[]; containment_rows=[]
for configuration in corpora:
    summary,pairs=redundancy(configuration); redundancy_rows.append(summary); containment_rows.extend(pairs)
write_csv(OUTPUT_DIR/"redundancy_summary.csv",redundancy_rows,list(redundancy_rows[0]))
write_csv(OUTPUT_DIR/"redundancy_containment_pairs.csv",containment_rows,list(containment_rows[0]) if containment_rows else ["configuration","card_key","left_chunk_id","right_chunk_id","left_unique_tokens","right_unique_tokens","containment_ratio"])
write_json(OUTPUT_DIR/"redundancy_summary.json",{"contract":REDUNDANCY_CONTRACT,"summaries":redundancy_rows})
normalization_rows=list(csv.DictReader((SOURCE_CHUNKS.parent/"retrieval_search_normalization_per_query.csv").open(encoding="utf-8",newline="")))
published_bm25={row["query_id"]:json.loads(row["top5_chunk_ids"]) for row in normalization_rows if row["configuration"]=="normalized_bm25"}
published_vector={row["query_id"]:json.loads(row["top5_chunk_ids"]) for row in all_baseline_rows if row["method"]=="vector"}
candidate_rows=list(csv.DictReader(SOURCE_16_CANDIDATES.open(encoding="utf-8",newline="")))
published_rrf={}
for query_id in gold_by_id:
    selected=sorted((row for row in candidate_rows if row["configuration"]=="vector_0.4_bm25_0.6" and row["query_id"]==query_id),key=lambda row:int(row["fused_rank"]))
    assert len(selected)==50; published_rrf[query_id]=[row["chunk_id"] for row in selected]
baseline_reproduction={"normalized_bm25_top5_exact_queries":sum(rankings[("existing_exact_v1",q)]["bm25"][:5]==published_bm25[q] for q in gold_by_id),"numpy_vector_top5_exact_queries":sum(rankings[("existing_exact_v1",q)]["vector"][:5]==published_vector[q] for q in gold_by_id),"notebook16_rrf_0.4_top5_exact_queries":sum(rankings[("existing_exact_v1",q)]["top50"][:5]==published_rrf[q][:5] for q in gold_by_id),"notebook16_rrf_0.4_top50_exact_queries":sum(rankings[("existing_exact_v1",q)]["top50"]==published_rrf[q] for q in gold_by_id),"baseline_name":"existing_exact_v1"}
baseline_reproduction["all_public_top5_exact"]=all(baseline_reproduction[key]==30 for key in ("normalized_bm25_top5_exact_queries","numpy_vector_top5_exact_queries","notebook16_rrf_0.4_top5_exact_queries"))
summary_by_key={(row["configuration"],row["group"]):row for row in summary_rows}
old_evidence=summary_by_key[("existing_exact_v1","evidence")]; new_evidence=summary_by_key[("structural_heading_path","evidence")]
old_all=summary_by_key[("existing_exact_v1","all")]; new_all=summary_by_key[("structural_heading_path","all")]
wlt_mrr=next(row for row in paired_wlt_rows if row["group"]=="evidence" and row["metric"]=="answer_mrr_at_5")
old_red=redundancy_rows[0]; new_red=redundancy_rows[1]
hit_improvements=sum(row["delta"]>0 for row in paired_delta_rows if row["group"]=="evidence" and row["metric"]=="answer_hit_at_3")
hit_nonreg=new_evidence["answer_hit_at_3"]>=old_evidence["answer_hit_at_3"]-1e-12
card_nonreg=new_all["card_hit_at_3"]>=old_all["card_hit_at_3"]-1e-12
improvement=hit_improvements>=1 or (abs(new_evidence["answer_hit_at_3"]-old_evidence["answer_hit_at_3"])<=1e-12 and new_evidence["answer_mrr_at_5"]-old_evidence["answer_mrr_at_5"]>=0.025-1e-12 and wlt_mrr["wins"]>wlt_mrr["losses"])
exact_nonworse=new_red["exact_duplicate_excess"]<=old_red["exact_duplicate_excess"]
containment_nonworse=new_red["containment_pair_count"]<=old_red["containment_pair_count"]
red_improved=new_red["exact_duplicate_excess"]<old_red["exact_duplicate_excess"] or new_red["containment_pair_count"]<old_red["containment_pair_count"]
gate_pass=hit_nonreg and card_nonreg and improvement and exact_nonworse and containment_nonworse and red_improved
selection_decision={"decision":"candidate_for_reranker_reevaluation_on_dev" if gate_pass else "do_not_advance_structural_chunking","gate_pass":gate_pass,"integrity_required":"PASS","evidence_hit_at_3_non_regression":hit_nonreg,"all_card_hit_at_3_non_regression":card_nonreg,"evidence_hit_improved_query_count":hit_improvements,"evidence_hit_tied":abs(new_evidence["answer_hit_at_3"]-old_evidence["answer_hit_at_3"])<=1e-12,"evidence_mrr_delta":new_evidence["answer_mrr_at_5"]-old_evidence["answer_mrr_at_5"],"evidence_mrr_wins":wlt_mrr["wins"],"evidence_mrr_losses":wlt_mrr["losses"],"improvement_condition":improvement,"exact_duplicate_nonworse":exact_nonworse,"containment_nonworse":containment_nonworse,"redundancy_one_strictly_improved":red_improved,"scope":"development_candidate_only_not_operations_or_holdout"}
write_json(OUTPUT_DIR/"selection_decision.json",selection_decision)
retrieval_summary={"experiment_name":EXPERIMENT_NAME,"scope":"development_30_queries_single_run","configurations":list(corpora),"settings":{"vector":"numpy_squared_l2","vector_tie":"chunk_id_lexical","bm25":{"k1":1.5,"b":0.75,"normalization":"notebook13 exact"},"rrf":{"vector_weight":0.4,"bm25_weight":0.6,"k":60,"depth":50},"excluded":["reranker","MMR","morphology","router","Chroma query"]},"ranking_freeze":ranking_freeze_manifest,"baseline_reproduction":baseline_reproduction,"summaries":summary_rows,"paired_wlt":paired_wlt_rows,"heading_inflation":{"relevant_rows_reviewed":len(heading_audit_rows),"heading_only_contribution_rows":sum(row["heading_only_contribution"] for row in heading_audit_rows)},"redundancy":{"contract":REDUNDANCY_CONTRACT,"summaries":redundancy_rows},"selection":selection_decision,"embedding":embedding_usage_22,"metric_guide_ko":{"Card Hit@3":"상위 3개 중 기대 카드 청크가 하나라도 있는 질의 비율","Card MRR@5":"상위 5개에서 기대 카드가 처음 나온 순위의 역수 평균","Evidence Hit@3":"상위 3개 중 카드와 필수 용어를 포함한 근거 청크가 있는 질의 비율","Evidence MRR@5":"상위 5개에서 level을 무시한 term-bearing 근거가 처음 나온 순위의 역수 평균","Recall/nDCG":"청크 분모 변화에 민감하여 gate에는 쓰지 않는 진단값"},"limitations":["Same 30 development queries are used for comparison.","Heading-path terms can supply terms absent from direct body; audited separately.","Recall and nDCG denominators change with chunk count.","Gate pass only nominates later reranker reevaluation."]}
write_json(OUTPUT_DIR/"retrieval_summary.json",retrieval_summary)
print(json.dumps({"baseline_reproduction":baseline_reproduction,"existing_evidence":{"hit3":old_evidence["answer_hit_at_3"],"mrr5":old_evidence["answer_mrr_at_5"]},"structural_evidence":{"hit3":new_evidence["answer_hit_at_3"],"mrr5":new_evidence["answer_mrr_at_5"]},"wlt_mrr":wlt_mrr,"redundancy":redundancy_rows,"decision":selection_decision["decision"]},ensure_ascii=False,indent=2))


{
  "baseline_reproduction": {
    "normalized_bm25_top5_exact_queries": 30,
    "numpy_vector_top5_exact_queries": 20,
    "notebook16_rrf_0.4_top5_exact_queries": 28,
    "notebook16_rrf_0.4_top50_exact_queries": 19,
    "baseline_name": "existing_exact_v1",
    "all_public_top5_exact": false
  },
  "existing_evidence": {
    "hit3": 0.9,
    "mrr5": 0.8791666666666667
  },
  "structural_evidence": {
    "hit3": 0.9,
    "mrr5": 0.875
  },
  "wlt_mrr": {
    "group": "evidence",
    "metric": "answer_mrr_at_5",
    "denominator": 20,
    "wins": 2,
    "losses": 1,
    "ties": 17,
    "mean_delta": -0.004166666666666663
  },
  "redundancy": [
    {
      "configuration": "existing_exact_v1",
      "chunk_count": 327,
      "exact_duplicate_group_count": 34,
      "exact_duplicate_excess": 41,
      "containment_pair_count": 459
    },
    {
      "configuration": "structural_heading_path",
      "chunk_count": 147,
      "exact_duplicate_group_count": 0,
      "exact_duplicate_excess

In [11]:
SOURCE_HASHES_FINAL=source_hash_map()
assert SOURCE_HASHES_FINAL==SOURCE_HASHES_BEFORE
assert raw_sha256(OUTPUT_DIR/"ranking_freeze.jsonl")==ranking_freeze_sha256
assert raw_sha256(OUTPUT_DIR/"embedding_target_manifest.json")==EMBEDDING_TARGET_MANIFEST_SHA256
readme=f"""# 22. 구조 기반 청킹 + 제목 경로 검색문

개발 질의 30개 single-run 결과입니다. raw 구조로 direct body 청크를 만들고 retrieval_text에 issuer/card/heading path를 붙였으며 evidence_text에는 인위적 issuer/card를 넣지 않았습니다.

- 구조 청크 {len(all_chunks)}개, hierarchy node {len(all_hierarchy)}개
- 신규 embedding {embedding_usage_22['items']}개, 실제 {embedding_usage_22['creation_api_input_tokens_total']} input tokens, 생성 요청 {embedding_usage_22['creation_api_requests_total']}회
- 공식 단가 계약 $0.02/1M input tokens, 계산 비용 USD {embedding_usage_22['creation_cost_usd']:.8f}
- 가격 출처: {embedding_usage_22['pricing_source_url']}
- 현재 fresh run API 요청 {embedding_usage_22['current_run_api_requests']}회
- 기존 query embedding 30개는 읽기 전용 cache 재사용
- Chroma query/reranker/MMR/morphology/router 0

Card Hit@3은 상위 3개 안에 기대 카드가 있는 비율, Card MRR@5는 기대 카드의 첫 순위 역수 평균입니다. Evidence Hit@3/MRR@5는 level을 무시하고 expected card와 required terms가 evidence_text에 있는 청크를 기준으로 합니다. Recall/nDCG는 청크 분모 변화 때문에 진단만 합니다. W/L/T는 같은 질의의 개선/하락/동률 수입니다.

ranking_freeze 파일은 gold 전에 저장한 양쪽 Top50입니다. relevance_sets와 heading_inflation_audit는 제목 경로 부풀림 수동 감사, redundancy 파일은 same-card exact/containment 감사입니다.

판정은 {selection_decision['decision']}입니다. 개발 후보 판단일 뿐 운영 또는 별도 일반화 결론이 아니며, 통과해도 다음 reranker 재평가 후보일 뿐입니다.
"""
(OUTPUT_DIR/"README.md").write_text(readme,encoding="utf-8")
excluded={"run_manifest.json","integrity.json"}
output_paths=sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file() and path.name not in excluded)
run_manifest={"experiment_name":EXPERIMENT_NAME,"input_raw_sha256":SOURCE_HASHES_BEFORE,"output_raw_sha256":{str(path.relative_to(OUTPUT_DIR)):raw_sha256(path) for path in output_paths},"self_hash_policy":"run_manifest.json, integrity.json, executed notebook excluded"}
write_json(OUTPUT_DIR/"run_manifest.json",run_manifest)
final_integrity={"status":"PASS","source_hashes_before":SOURCE_HASHES_BEFORE,"source_hashes_after":SOURCE_HASHES_FINAL,"source_unchanged":SOURCE_HASHES_BEFORE==SOURCE_HASHES_FINAL,"old_cache_files_unchanged":all(SOURCE_HASHES_FINAL[str(path.relative_to(ROOT))]==SOURCE_HASHES_BEFORE[str(path.relative_to(ROOT))] for path in CACHE_FILES),"chroma_files_unchanged":all(SOURCE_HASHES_FINAL[str(path.relative_to(ROOT))]==SOURCE_HASHES_BEFORE[str(path.relative_to(ROOT))] for path in CHROMA_FILES),"chroma_client_opened":False,"chroma_queries":0,"embedding":{"new_count":len(new_vectors_by_chunk_id),"dimension":1536,"dtype":"float32","finite":bool(np.isfinite(new_matrix).all()),"creation_api_requests":creation_api_requests_total,"creation_api_input_tokens":creation_api_input_tokens_total,"current_run_api_requests":current_run_api_requests,"other_network_requests":0,"secret_persisted":False},"ranking":{"freeze_rows":len(ranking_freeze_rows),"depth":50,"unique_per_query":True,"rrf_score_formula_verified":True,"tie_break_verified":True,"gold_loaded_before_freeze":False},"evaluation":{"per_query_rows":len(per_query_rows),"summary_rows":len(summary_rows),"relevance_rows":len(relevance_rows),"changed_top5_rows":len(changed_rows),"metrics_in_range":all(value is None or 0<=value<=1 for row in per_query_rows for key,value in row.items() if key in {"card_hit_at_3","card_mrr_at_5","answer_hit_at_3","answer_recall_at_5","answer_mrr_at_5","answer_ndcg_at_5"})},"baseline_reproduction":baseline_reproduction,"selection":selection_decision,"ranking_freeze_raw_sha256":ranking_freeze_sha256,"embedding_target_manifest_raw_sha256":EMBEDDING_TARGET_MANIFEST_SHA256,"run_manifest_raw_sha256":raw_sha256(OUTPUT_DIR/"run_manifest.json")}
assert final_integrity["source_unchanged"] and final_integrity["old_cache_files_unchanged"] and final_integrity["chroma_files_unchanged"]
assert final_integrity["embedding"]["finite"] and final_integrity["evaluation"]["metrics_in_range"]
write_json(OUTPUT_DIR/"integrity.json",final_integrity)
saved_manifest=json.loads((OUTPUT_DIR/"run_manifest.json").read_text(encoding="utf-8"))
assert {relative:raw_sha256(OUTPUT_DIR/relative) for relative in saved_manifest["output_raw_sha256"]}==saved_manifest["output_raw_sha256"]
assert source_hash_map()==SOURCE_HASHES_BEFORE
print({"integrity":"PASS","source_unchanged":True,"current_run_api_requests":current_run_api_requests,"manifest_files":len(run_manifest["output_raw_sha256"]),"decision":selection_decision["decision"]})


{'integrity': 'PASS', 'source_unchanged': True, 'current_run_api_requests': 0, 'manifest_files': 31, 'decision': 'candidate_for_reranker_reevaluation_on_dev'}


## Independent review disposition — 사전 gate와 최종 개발 판단의 분리

독립 검증은 ranking, metric, hash 계산을 PASS로 확인했습니다. 기존 사전 기술 gate는 Evidence Hit@3에서 1 win/1 loss로 macro 0.90→0.90 동률이고 Evidence MRR@5는 0.8791667→0.875(-0.0041667)였지만, 최소 1 Hit win 절 때문에 technical gate_pass=true가 됐습니다. 이 사전 계약과 계산 결과는 소급 변경하지 않습니다.

독립 검토는 이 selection strength가 부적절하고 외부 16 기준선도 30/20/28/19로 완전 재현되지 않았다고 판단했습니다. 따라서 기술 gate 통과와 별개로 최종 개발 disposition은 inconclusive_keep_existing_baseline_and_retain_structural_candidate_for_followup입니다. 운영 적용, 별도 일반화 판단, 현재 청킹 교체는 허용하지 않습니다. 더 엄격한 gate는 후속 실험 결과를 보기 전에 새로 선언하고 이번 결과에 소급 적용하지 않습니다.

In [12]:
assert current_run_api_requests==0
assert creation_api_requests_total==3 and creation_api_input_tokens_total==65856
assert selection_decision["gate_pass"] is True
assert selection_decision["evidence_hit_improved_query_count"]==1
assert selection_decision["evidence_hit_tied"] is True
assert abs(old_evidence["answer_hit_at_3"]-0.90)<=1e-12
assert abs(new_evidence["answer_hit_at_3"]-0.90)<=1e-12
assert abs(old_evidence["answer_mrr_at_5"]-0.8791666666666667)<=1e-12
assert abs(new_evidence["answer_mrr_at_5"]-0.875)<=1e-12
assert abs(selection_decision["evidence_mrr_delta"]-(-0.004166666666666652))<=1e-12
assert baseline_reproduction=={
 "normalized_bm25_top5_exact_queries":30,
 "numpy_vector_top5_exact_queries":20,
 "notebook16_rrf_0.4_top5_exact_queries":28,
 "notebook16_rrf_0.4_top50_exact_queries":19,
 "baseline_name":"existing_exact_v1",
 "all_public_top5_exact":False}
ORIGINAL_CORE_HASHES={
 "evaluation_contract.json":"e9ba622ad15ce78eaff4fd33755a677e67b5172ef33343421ac926e6281072d2",
 "selection_decision.json":"ac5ebc127d8b379897694d622302166320385db39ff15a621c14d063a3cbd0c8",
 "retrieval_per_query.csv":"4ea068584490d3e3c633236f3e15e3e3b110c2f267770b198bd8578637d706d5",
 "retrieval_summary.csv":"24fe5e79c54ee6249faa09d0622086c06025aba5192dce9480f9749c99acc35c",
 "ranking_freeze.jsonl":"e7d6b5db7fe9852f131e597d2b2aa198a9125f34fdac2674e3fa0f3358192734",
 "redundancy_summary.json":"7628d39a79c2e67e3f1acfdcebbd1b9c8f47dc6b6b0f9a45046311febd01716d"}
assert {name:raw_sha256(OUTPUT_DIR/name) for name in ORIGINAL_CORE_HASHES}==ORIGINAL_CORE_HASHES
independent_review_disposition={
 "independent_verification":{"rankings":"PASS","metrics":"PASS","hashes":"PASS"},
 "technical_predeclared_gate":"technical_predeclared_gate_pass_but_not_accepted_for_selection",
 "technical_gate_pass_preserved":True,
 "technical_gate_observation":{
   "evidence_hit_at_3_baseline":0.90,
   "evidence_hit_at_3_structural":0.90,
   "paired_hit_wins":1,
   "paired_hit_losses":1,
   "evidence_mrr_at_5_baseline":0.8791666666666667,
   "evidence_mrr_at_5_structural":0.875,
   "evidence_mrr_delta":-0.004166666666666652,
   "gate_weakness":"at_least_one_hit_win_clause_passed_despite_one_hit_loss_and_negative_mrr_delta"},
 "external_baseline_reproduction":baseline_reproduction,
 "final_development_disposition":"inconclusive_keep_existing_baseline_and_retain_structural_candidate_for_followup",
 "allowed_use":"followup_research_candidate_only",
 "prohibited_conclusions":["operations_adoption","holdout_generalization","replace_current_chunking"],
 "future_gate_policy":"Declare a stricter gate before a new follow-up experiment; do not retroactively apply it to this result.",
 "historical_embedding_usage":{"creation_api_requests":3,"creation_api_input_tokens":65856},
 "current_run":{"api_requests":0,"external_network_requests":0,"chroma_queries":0,"gpu":0},
 "original_contract_and_selection_unchanged":True,
 "original_core_raw_sha256":ORIGINAL_CORE_HASHES}
write_json(OUTPUT_DIR/"independent_review_disposition.json",independent_review_disposition)
readme_path=OUTPUT_DIR/"README.md"
readme_text=readme_path.read_text(encoding="utf-8")
review_section="""
## 독립 검토 최종 disposition

계산·ranking·hash 검증은 PASS지만 사전 기술 gate의 선택 강도는 채택 근거로 부족했습니다. Evidence Hit@3는 1개 개선과 1개 하락으로 0.90→0.90 동률이고, MRR@5는 0.8791667→0.875로 0.0041667 하락했습니다. 그런데 사전의 최소 1 Hit win 절 때문에 기술 gate는 통과했습니다. 기존 gate_pass=true와 계약은 소급 변경하지 않습니다.

최종 개발 판단은 inconclusive_keep_existing_baseline_and_retain_structural_candidate_for_followup입니다. 기존 기준선을 유지하고 구조 청킹은 후속 연구 후보로만 남깁니다. 운영 적용, 별도 일반화 결론, 현재 청킹 교체는 허용하지 않습니다. 더 엄격한 gate는 후속 실험 전에 새로 선언하며 이번 결과를 소급 재판정하지 않습니다.
"""
assert "## 독립 검토 최종 disposition" not in readme_text
readme_path.write_text(readme_text.rstrip()+"\n"+review_section,encoding="utf-8")
excluded={"run_manifest.json","integrity.json"}
output_paths=sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file() and path.name not in excluded)
run_manifest={"experiment_name":EXPERIMENT_NAME,"input_raw_sha256":SOURCE_HASHES_BEFORE,
 "output_raw_sha256":{str(path.relative_to(OUTPUT_DIR)):raw_sha256(path) for path in output_paths},
 "notebook_raw_sha256":None,
 "notebook_hash_finalization":"finalized_after_nbconvert",
 "self_hash_policy":"run_manifest.json and integrity.json excluded"}
write_json(OUTPUT_DIR/"run_manifest.json",run_manifest)
final_integrity=json.loads((OUTPUT_DIR/"integrity.json").read_text(encoding="utf-8"))
final_integrity["independent_review_disposition"]=independent_review_disposition
final_integrity["current_run_api_requests_zero_asserted"]=True
final_integrity["original_core_hashes_exact"]=True
final_integrity["notebook_raw_sha256"]=None
final_integrity["notebook_hash_finalization"]="finalized_after_nbconvert"
final_integrity["run_manifest_raw_sha256"]=raw_sha256(OUTPUT_DIR/"run_manifest.json")
write_json(OUTPUT_DIR/"integrity.json",final_integrity)
assert source_hash_map()==SOURCE_HASHES_BEFORE
assert json.loads((OUTPUT_DIR/"embedding_usage.json").read_text(encoding="utf-8"))["current_run_api_requests"]==0
print({"independent_verification":"PASS","technical_gate_pass_preserved":True,
 "final_development_disposition":independent_review_disposition["final_development_disposition"],
 "current_run_api_requests":0,"original_core_hashes_exact":True})


{'independent_verification': 'PASS', 'technical_gate_pass_preserved': True, 'final_development_disposition': 'inconclusive_keep_existing_baseline_and_retain_structural_candidate_for_followup', 'current_run_api_requests': 0, 'original_core_hashes_exact': True}


## Follow-up 1 — operational-prototype baseline scope correction

이 비교는 기존 결과를 본 뒤 발견한 baseline 범위 누락을 보완하는 post-hoc diagnostic입니다. 기존 formal experiment, contract, selection, independent disposition을 변경하거나 섞지 않습니다. current prototype처럼 existing_exact_v1의 fused Top50에서 level section/benefit만 순서 보존 필터한 baseline과 structural_heading_path의 기존 Top50을 비교합니다. Confirmatory/promotion gate가 아니며 formal promotion은 불가능합니다.

In [13]:
assert current_run_api_requests==0
assert creation_api_requests_total==3 and creation_api_input_tokens_total==65856
F1_FROZEN_ORIGINAL_HASHES={
 "evaluation_contract.json":"e9ba622ad15ce78eaff4fd33755a677e67b5172ef33343421ac926e6281072d2",
 "selection_decision.json":"ac5ebc127d8b379897694d622302166320385db39ff15a621c14d063a3cbd0c8",
 "retrieval_per_query.csv":"4ea068584490d3e3c633236f3e15e3e3b110c2f267770b198bd8578637d706d5",
 "retrieval_summary.csv":"24fe5e79c54ee6249faa09d0622086c06025aba5192dce9480f9749c99acc35c",
 "ranking_freeze.jsonl":"e7d6b5db7fe9852f131e597d2b2aa198a9125f34fdac2674e3fa0f3358192734",
 "redundancy_summary.json":"7628d39a79c2e67e3f1acfdcebbd1b9c8f47dc6b6b0f9a45046311febd01716d",
 "independent_review_disposition.json":"ce1f1ce985733b482da3c7c38971bb3af4b40deda16c0aa60cb9a35ad1d1f990",
 "retrieval_summary.json":"979a04b0407193dfa4c48ec28697ae339b0e3072c31e54e6bc02edd8d427fc03",
 "paired_deltas.csv":"6143cd1b1f1cba11eb51ee847b44a6e1eb602d18c9204f6ae639a515b5d27f61",
 "paired_wlt.csv":"437976a9c87321899abbd1d2926db5a0cc26f7aeecb2faffc8aa0d1eef5b15f5",
 "changed_top5.csv":"10d540d2379240bce3bf15659f71d5804964dcec3064ef3f56a160de976b6264",
 "relevance_sets.jsonl":"f5d98929fb8c591f84d09727036d94dd40da0f880903057dd2cc5bfe7b21e6e2",
 "heading_inflation_audit.csv":"0b9b3bb7c3611fc3d17266b86828280ede5c76535e5b68935cfc2f337ffaac8a",
 "redundancy_summary.csv":"fd37b8e48f94f7bd85c18f37678cffa3e033aee97584a642d6297e0637a68f66",
 "redundancy_containment_pairs.csv":"a767254883ae43bdb7773a35574751af72425176b177c8958a185752c3083385"}
assert {name:raw_sha256(OUTPUT_DIR/name) for name in F1_FROZEN_ORIGINAL_HASHES}==F1_FROZEN_ORIGINAL_HASHES
prototype_path=ROOT/"test_scripts/search_retrieval/pipeline.py"
prototype_hash_before=raw_sha256(prototype_path)
assert prototype_hash_before=="9cfc9acc9d00613abbd5d1a595b109f6584840fa94fb4eabab2b96f627f59bbd"
prototype_source=prototype_path.read_text(encoding="utf-8")
for required_source in (
 'LEAF_LEVELS = frozenset(("section", "benefit"))',
 'for candidate in candidates:',
 'if chunk["metadata"]["level"] in LEAF_LEVELS:',
 'result.append(dict(candidate))',
 'candidates[:component_depth]',
):
    assert required_source in prototype_source
LEAF_LEVELS_F1={"section","benefit"}
f1_old_rankings={}
f1_candidate_rows=[]
for query_id in gold_by_id:
    original=rankings[("existing_exact_v1",query_id)]["top50"]
    filtered=[identifier for identifier in original if existing_by_id[identifier]["metadata"]["level"] in LEAF_LEVELS_F1]
    reference=[identifier for identifier in original if existing_by_id[identifier]["metadata"]["level"] in {"section","benefit"}]
    assert filtered==reference and all(item in original for item in filtered)
    assert [original.index(item) for item in filtered]==sorted(original.index(item) for item in filtered)
    assert len(filtered)>=5
    f1_old_rankings[query_id]=filtered
    f1_candidate_rows.append({
      "query_id":query_id,"old_top50_count":50,"old_available_leaf_count":len(filtered),
      "old_leaf_top5_chunk_ids":canonical_json(filtered[:5]),
      "old_leaf_top5_levels":canonical_json([existing_by_id[item]["metadata"]["level"] for item in filtered[:5]]),
      "new_top50_count":50,
      "new_top5_chunk_ids":canonical_json(rankings[("structural_heading_path",query_id)]["top50"][:5]),
      "prototype_filter_order_exact":True})
assert len(f1_candidate_rows)==30
write_csv(OUTPUT_DIR/"followup1_candidate_rankings.csv",f1_candidate_rows,list(f1_candidate_rows[0]))
f1_per_query=[]
f1_relevant_counts=[]
for configuration in ("existing_exact_v1_leaf_filtered","structural_heading_path"):
    for query_id,gold in gold_by_id.items():
        if configuration=="existing_exact_v1_leaf_filtered":
            ranking=f1_old_rankings[query_id]
            metric_configuration="existing_exact_v1"
        else:
            ranking=rankings[("structural_heading_path",query_id)]["top50"]
            metric_configuration="structural_heading_path"
        metrics=evaluate_query(metric_configuration,gold,ranking)
        f1_per_query.append({
          "configuration":configuration,"query_id":query_id,
          "question_group":"card" if gold["expected_level"]=="card" else "evidence",
          "category":gold["category"],"candidate_count":len(ranking),**metrics,
          "top5_chunk_ids":canonical_json(ranking[:5]),
          "top5_cards":canonical_json([card_key_for(metric_configuration,item) for item in ranking[:5]])})
        f1_relevant_counts.append({
          "configuration":configuration,"query_id":query_id,
          "question_group":"card" if gold["expected_level"]=="card" else "evidence",
          "category":gold["category"],
          "answer_relevant_count":metrics["answer_relevant_count"]})
assert len(f1_per_query)==60 and len(f1_relevant_counts)==60
write_csv(OUTPUT_DIR/"followup1_per_query.csv",f1_per_query,list(f1_per_query[0]))
write_csv(OUTPUT_DIR/"followup1_relevant_counts.csv",f1_relevant_counts,list(f1_relevant_counts[0]))
f1_summary=[]
for configuration in ("existing_exact_v1_leaf_filtered","structural_heading_path"):
    rows=[row for row in f1_per_query if row["configuration"]==configuration]
    groups={"all":rows,"card":[row for row in rows if row["question_group"]=="card"],
            "evidence":[row for row in rows if row["question_group"]=="evidence"],
            "numeric":[row for row in rows if row["category"]=="numeric_condition"],
            "semantic":[row for row in rows if row["category"]=="semantic"]}
    f1_summary.extend({"configuration":configuration,"group":group,"denominator":len(selected),**aggregate(selected)} for group,selected in groups.items())
assert len(f1_summary)==10
write_csv(OUTPUT_DIR/"followup1_summary.csv",f1_summary,list(f1_summary[0]))
f1_by_key={(row["configuration"],row["query_id"]):row for row in f1_per_query}
f1_deltas=[]; f1_wlt=[]
for group in ("all","card","evidence","numeric","semantic"):
    eligible=[qid for qid,gold in gold_by_id.items() if group=="all" or (group=="card" and gold["expected_level"]=="card") or (group=="evidence" and gold["expected_level"]!="card") or (group=="numeric" and gold["category"]=="numeric_condition") or (group=="semantic" and gold["category"]=="semantic")]
    for metric in ("card_hit_at_3","card_mrr_at_5","answer_hit_at_3","answer_recall_at_5","answer_mrr_at_5","answer_ndcg_at_5"):
        values=[]
        for qid in eligible:
            old=f1_by_key[("existing_exact_v1_leaf_filtered",qid)][metric]
            new=f1_by_key[("structural_heading_path",qid)][metric]
            if old is not None and new is not None: values.append((qid,old,new))
        if not values: continue
        wins=sum(new>old+1e-12 for _,old,new in values); losses=sum(new<old-1e-12 for _,old,new in values)
        ties=len(values)-wins-losses
        f1_deltas.extend({"group":group,"metric":metric,"query_id":qid,"baseline":old,"structural":new,"delta":new-old} for qid,old,new in values)
        f1_wlt.append({"group":group,"metric":metric,"denominator":len(values),"wins":wins,"losses":losses,"ties":ties,"mean_delta":sum(new-old for _,old,new in values)/len(values)})
write_csv(OUTPUT_DIR/"followup1_paired_deltas.csv",f1_deltas,list(f1_deltas[0]))
write_csv(OUTPUT_DIR/"followup1_wlt.csv",f1_wlt,list(f1_wlt[0]))
f1_changed=[]
for query_id,gold in gold_by_id.items():
    old=f1_old_rankings[query_id][:5]; new=rankings[("structural_heading_path",query_id)]["top50"][:5]
    if old!=new:
        old_metrics=evaluate_query("existing_exact_v1",gold,old)
        new_metrics=evaluate_query("structural_heading_path",gold,new)
        f1_changed.append({"query_id":query_id,"category":gold["category"],
          "question_group":"card" if gold["expected_level"]=="card" else "evidence",
          "old_top5_chunk_ids":canonical_json(old),"new_top5_chunk_ids":canonical_json(new),
          "card_hit_delta":new_metrics["card_hit_at_3"]-old_metrics["card_hit_at_3"],
          "card_mrr_delta":new_metrics["card_mrr_at_5"]-old_metrics["card_mrr_at_5"],
          "answer_hit_delta":None if old_metrics["answer_hit_at_3"] is None else new_metrics["answer_hit_at_3"]-old_metrics["answer_hit_at_3"],
          "answer_mrr_delta":None if old_metrics["answer_mrr_at_5"] is None else new_metrics["answer_mrr_at_5"]-old_metrics["answer_mrr_at_5"]})
write_csv(OUTPUT_DIR/"followup1_changed_top5.csv",f1_changed,list(f1_changed[0]))
f1_summary_by_key={(row["configuration"],row["group"]):row for row in f1_summary}
f1_disposition={
 "experiment_type":"post_hoc_baseline_scope_correction_diagnostic",
 "formal_promotion_eligible":False,
 "formal_selection_decision_unchanged":True,
 "current_baseline_disposition":"retain_existing_operational_prototype_baseline_pending_confirmatory_test",
 "structural_candidate_status":"retain_for_followup_only",
 "comparison":{"baseline":"existing_exact_v1_leaf_filtered","challenger":"structural_heading_path"},
 "prototype_contract":{"source_path":str(prototype_path.relative_to(ROOT)),"source_raw_sha256":prototype_hash_before,
   "fused_depth":50,"leaf_levels":["benefit","section"],"order_preserved":True,"static_contract_exact":True},
 "metrics":{"summaries":f1_summary,"paired_wlt":f1_wlt},
 "inherited_limitation":{"public_notebook16_reproduction":baseline_reproduction,
   "note":"existing_exact_v1 Top50 is not fully identical to historical notebook16 0.4 Top50"},
 "interpretation":"Diagnostic direct comparison only; no confirmatory or promotion gate was declared before observing this follow-up.",
 "prohibited":["formal_promotion","operations_adoption","holdout_generalization","replace_current_chunking"]}
write_json(OUTPUT_DIR/"followup1_disposition.json",f1_disposition)
write_json(OUTPUT_DIR/"followup1_summary.json",{
 "experiment_type":f1_disposition["experiment_type"],"summaries":f1_summary,
 "paired_wlt":f1_wlt,"candidate_count":{"min":min(row["old_available_leaf_count"] for row in f1_candidate_rows),
 "median":statistics.median(row["old_available_leaf_count"] for row in f1_candidate_rows),
 "max":max(row["old_available_leaf_count"] for row in f1_candidate_rows)},
 "changed_top5_queries":len(f1_changed),"disposition":f1_disposition["current_baseline_disposition"]})
assert {name:raw_sha256(OUTPUT_DIR/name) for name in F1_FROZEN_ORIGINAL_HASHES}==F1_FROZEN_ORIGINAL_HASHES
assert raw_sha256(prototype_path)==prototype_hash_before
readme_path=OUTPUT_DIR/"README.md"; readme_text=readme_path.read_text(encoding="utf-8")
followup_section="""
## Follow-up 1 — operational-prototype baseline scope correction

이 비교는 결과 후 발견한 baseline 범위 누락을 보완한 post-hoc diagnostic입니다. 기존 327개 RRF Top50에서 section/benefit만 순서 보존 필터한 current prototype 범위와 구조 청킹 Top50을 직접 비교합니다. 기존 formal selection과 independent disposition은 바꾸지 않습니다.

결과는 followup1 전용 파일에만 저장했습니다. confirmatory/promotion gate가 아니므로 formal promotion은 불가능합니다. 현재 disposition은 retain_existing_operational_prototype_baseline_pending_confirmatory_test이며 구조 청킹은 후속 후보로만 남깁니다. 기존 exact-v1과 historical notebook16 0.4 Top50이 완전 일치하지 않는 한계도 그대로 상속합니다.
"""
assert "## Follow-up 1 — operational-prototype baseline scope correction" not in readme_text
readme_path.write_text(readme_text.rstrip()+"\n"+followup_section,encoding="utf-8")
excluded={"run_manifest.json","integrity.json"}
output_paths=sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file() and path.name not in excluded)
run_manifest={"experiment_name":EXPERIMENT_NAME,"input_raw_sha256":SOURCE_HASHES_BEFORE,
 "output_raw_sha256":{str(path.relative_to(OUTPUT_DIR)):raw_sha256(path) for path in output_paths},
 "notebook_raw_sha256":None,"notebook_hash_finalization":"finalized_after_nbconvert",
 "self_hash_policy":"run_manifest.json and integrity.json excluded"}
write_json(OUTPUT_DIR/"run_manifest.json",run_manifest)
final_integrity=json.loads((OUTPUT_DIR/"integrity.json").read_text(encoding="utf-8"))
final_integrity["followup1"]={"status":"PASS","per_query_rows":len(f1_per_query),"summary_rows":len(f1_summary),
 "candidate_rows":len(f1_candidate_rows),"changed_top5_rows":len(f1_changed),
 "prototype_source_raw_sha256":prototype_hash_before,"prototype_source_unchanged":True,
 "original_result_hashes_exact":True,"current_run_api_requests":0,
 "disposition":f1_disposition["current_baseline_disposition"]}
final_integrity["notebook_raw_sha256"]=None
final_integrity["notebook_hash_finalization"]="finalized_after_nbconvert"
final_integrity["run_manifest_raw_sha256"]=raw_sha256(OUTPUT_DIR/"run_manifest.json")
write_json(OUTPUT_DIR/"integrity.json",final_integrity)
assert source_hash_map()==SOURCE_HASHES_BEFORE and current_run_api_requests==0
print({"followup1":"PASS","per_query_rows":60,"summary_rows":10,
 "candidate_count_min":min(row["old_available_leaf_count"] for row in f1_candidate_rows),
 "candidate_count_max":max(row["old_available_leaf_count"] for row in f1_candidate_rows),
 "changed_top5":len(f1_changed),"disposition":f1_disposition["current_baseline_disposition"]})


{'followup1': 'PASS', 'per_query_rows': 60, 'summary_rows': 10, 'candidate_count_min': 30, 'candidate_count_max': 42, 'changed_top5': 30, 'disposition': 'retain_existing_operational_prototype_baseline_pending_confirmatory_test'}


## Follow-up 2 — 새 구조 BM25/RRF 소규모 grid (append-only 실행)

기존 셀은 재실행하지 않습니다. 아래 self-contained 셀만 별도 fresh kernel에서 저장된 chunks/cache/query/Follow-up 1을 직접 로드합니다. 결과 전 27개 grid와 guardrail·선택 tie-break를 고정하며 API·network·embedding·GPU·Chroma query는 0입니다. Recall/nDCG는 진단값이고 통과 조합도 `exploratory_candidate_only`입니다.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from decimal import Decimal
import csv, hashlib, itertools, json, math, os, re, resource, time, unicodedata
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert ROOT.name == "PickCardU"
assert os.getenv("RUN_APPROVED_22_EMBEDDING", "0") == "0"
OUT = ROOT / "notebooks/data/22_structural_heading_chunking_ablation"
QUERY_CSV = ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/retrieval_per_query.csv"
OLD_CHUNKS = ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl"
NEW_CHUNKS = OUT / "chunks.jsonl"
TARGET_MANIFEST = OUT / "embedding_target_manifest.json"
EMBEDDING_USAGE = OUT / "embedding_usage.json"
RANKING_FREEZE = OUT / "ranking_freeze.jsonl"
F1_PER_QUERY = OUT / "followup1_per_query.csv"
F1_SUMMARY = OUT / "followup1_summary.json"
OLD_CACHE_ROOT = ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/embedding_cache/text-embedding-3-small"
OLD_CACHE_FILES = [OLD_CACHE_ROOT / (name + ".npz") for name in [
 "3c81bda6bc8b1e69ca305b0dcc219203baeede6797e91045dbde36244babc3b4",
 "335a583c624bd1fe61aec74282af4e1462af20962c96147ada05ed5a29605fba",
 "93cd3f7a2ca8c9e070421705a575465a92f29966aebae38a583c2da60a6ee97f",
 "b6ae13c54bbba3d250e83c39da8673647b077dfc91f671565b4cef016593fbf2",
 "8433343533d6e8ae60f40c42d7e4a0b43b751bb36e26532e738ff51eac4a9ae0",
 "cb5d9e87e091be498d91fa51d46619ca60c7fe5314b27a6686b017cb8c304e55"]]
PROTECTED_EXISTING = [OUT / name for name in (
 "evaluation_contract.json","selection_decision.json","independent_review_disposition.json",
 "retrieval_per_query.csv","retrieval_summary.csv","retrieval_summary.json","ranking_freeze.jsonl",
 "followup1_candidate_rankings.csv","followup1_changed_top5.csv","followup1_disposition.json",
 "followup1_paired_deltas.csv","followup1_per_query.csv","followup1_relevant_counts.csv",
 "followup1_summary.csv","followup1_summary.json","followup1_wlt.csv")]

def raw_sha256(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def canonical_json(value): return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(",",":"))
def write_json(path,value): path.write_text(json.dumps(value,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
def write_csv(path,rows,fields):
    with path.open("w",encoding="utf-8",newline="") as handle:
        writer=csv.DictWriter(handle,fieldnames=fields); writer.writeheader(); writer.writerows(rows)

usage=json.loads(EMBEDDING_USAGE.read_text(encoding="utf-8"))
assert (usage["items"],usage["dimension"],usage["finite"])==(147,1536,True)
assert (usage["creation_api_requests_total"],usage["creation_api_input_tokens_total"],usage["current_run_api_requests"])==(3,65856,0)
new_cache_files=[]
for batch in usage["batches"]:
    path=ROOT/batch["cache_path"]; assert path.parent==OUT/"embedding_cache/text-embedding-3-small"
    new_cache_files.append(path)
assert len(new_cache_files)==3
READ_ONLY_INPUTS=list(dict.fromkeys([QUERY_CSV,OLD_CHUNKS,NEW_CHUNKS,TARGET_MANIFEST,EMBEDDING_USAGE,
 RANKING_FREEZE,F1_PER_QUERY,F1_SUMMARY,*OLD_CACHE_FILES,*new_cache_files,*PROTECTED_EXISTING]))
input_hashes_before={str(path.relative_to(ROOT)):raw_sha256(path) for path in READ_ONLY_INPUTS}

F2_K1_VALUES=[1.2,1.5,1.8]; F2_B_VALUES=[.5,.75,1.0]
F2_WEIGHTS=[(.3,.7),(.4,.6),(.5,.5)]
F2_CONFIGS=[{"grid_order":i,"configuration":f"k1_{k1:g}_b_{b:g}_vector_{vw:g}_bm25_{bw:g}",
 "k1":k1,"b":b,"vector_weight":vw,"bm25_weight":bw}
 for i,(k1,b,(vw,bw)) in enumerate(itertools.product(F2_K1_VALUES,F2_B_VALUES,F2_WEIGHTS))]
assert len(F2_CONFIGS)==len({row["configuration"] for row in F2_CONFIGS})==27
F2_GUARDRAILS={
 "evidence":{"answer_hit_at_3":.90,"answer_mrr_at_5":.8891666666666667,
             "card_hit_at_3":.90,"card_mrr_at_5":.8891666666666667},
 "numeric":{"answer_hit_at_3":1.0,"answer_mrr_at_5":.9333333333333333},
 "semantic":{"answer_hit_at_3":.80,"answer_mrr_at_5":.845},
 "card":{"card_hit_at_3":1.0,"card_mrr_at_5":.9333333333333333}}
F2_PRIORITY=["evidence_answer_hit_at_3","numeric_answer_hit_at_3","evidence_answer_mrr_at_5",
 "numeric_answer_mrr_at_5","semantic_answer_mrr_at_5","card_card_mrr_at_5"]
stored_f1_summary=json.loads(F1_SUMMARY.read_text(encoding="utf-8"))["summaries"]
stored_f1_by_key={(row["configuration"],row["group"]):row for row in stored_f1_summary}
for group,metrics in F2_GUARDRAILS.items():
    stored=stored_f1_by_key[("existing_exact_v1_leaf_filtered",group)]
    for metric,expected in metrics.items(): assert abs(stored[metric]-expected)<=1e-12

F2_CONTRACT={"declared_before_results":True,
 "execution_mode":"append_only_cells_in_fresh_kernel_existing_cells_not_reexecuted",
 "experiment_type":"structural_heading_chunking_bm25_rrf_grid_followup",
 "scope":"same_30_development_queries_cached_embeddings_only",
 "grid":{"k1":F2_K1_VALUES,"b":F2_B_VALUES,"vector_bm25_weights":[list(x) for x in F2_WEIGHTS],
         "configuration_count":27},
 "fixed":{"rrf_k":60,"component_depth":50,"fused_depth":50,"evaluation_top_k":[3,5],
          "vector_distance":"numpy_squared_l2","score_tie_break":"chunk_id_lexical",
          "excluded_changes":["morphology","MMR","reranker","depth","rrf_k"]},
 "baseline":{"name":"existing_exact_v1_leaf_filtered","source":"Follow-up 1 stored results",
             "guardrails":F2_GUARDRAILS,"exact_reproduction_required":True},
 "default_structural":{"k1":1.5,"b":.75,"vector_weight":.4,"bm25_weight":.6,
                       "exact_reproduction_required":True},
 "selection":{"priority_descending":F2_PRIORITY,
              "default_distance":"abs(k1-1.5)+abs(b-0.75)+abs(vector_weight-0.4)+abs(bm25_weight-0.6), ascending",
              "final_tie_break":"predeclared grid_order ascending",
              "if_none":"retain_existing_operational_prototype","if_any":"exploratory_candidate_only",
              "promotion_eligible":False},
 "diagnostic_only":["answer_recall_at_5","answer_ndcg_at_5"],
 "execution":{"environment":"skn25","new_api_requests":0,"new_embeddings":0,
              "external_network_requests":0,"gpu":0,"chroma_queries":0,
              "index_storage_changes":0,"api_cost_usd":0.0}}
write_json(OUT/"followup2_contract.json",F2_CONTRACT)
print({"contract":"fixed_before_results","configurations":27,"execution_mode":F2_CONTRACT["execution_mode"]})


{'contract': 'fixed_before_results', 'configurations': 27, 'execution_mode': 'append_only_cells_in_fresh_kernel_existing_cells_not_reexecuted'}


In [2]:
f2_wall_start=time.perf_counter(); f2_cpu_start=time.process_time()
f2_rss_start=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss

query_rows={}
with QUERY_CSV.open(encoding="utf-8",newline="") as handle:
    for row in csv.DictReader(handle):
        if row["method"]=="keyword":
            query_rows[row["query_id"]]={"query":row["query"],"category":row["category"],
             "expected_card":row["expected_card"],"expected_level":row["expected_level"],
             "required_terms":json.loads(row["required_terms"])}
assert len(query_rows)==30
assert Counter(row["category"] for row in query_rows.values())=={"proper_noun":10,"numeric_condition":10,"semantic":10}
assert sum(row["expected_level"]=="card" for row in query_rows.values())==10
new_chunks=[json.loads(line) for line in NEW_CHUNKS.read_text(encoding="utf-8").splitlines() if line]
new_by_id={row["chunk_id"]:row for row in new_chunks}
assert len(new_chunks)==len(new_by_id)==147
new_ids=[row["chunk_id"] for row in new_chunks]

new_vector_count=0
for path in new_cache_files:
    with np.load(path,allow_pickle=False) as cached:
        matrix=cached["embeddings"]; new_vector_count+=len(matrix)
        assert matrix.ndim==2 and matrix.shape[1]==1536 and matrix.dtype==np.float32 and np.isfinite(matrix).all()
assert new_vector_count==147
old_vector_count=0
for path in OLD_CACHE_FILES:
    with np.load(path,allow_pickle=False) as cached:
        matrix=cached["embeddings"]; old_vector_count+=len(matrix)
        assert matrix.ndim==2 and matrix.shape[1]==1536 and matrix.dtype==np.float32 and np.isfinite(matrix).all()
assert old_vector_count==357

saved_freeze={}
for line in RANKING_FREEZE.read_text(encoding="utf-8").splitlines():
    if line:
        row=json.loads(line)
        if row["configuration"]=="structural_heading_path": saved_freeze[row["query_id"]]=row
assert len(saved_freeze)==30
vectors={qid:row["vector_top50_chunk_ids"] for qid,row in saved_freeze.items()}
assert all(len(row)==len(set(row))==50 and set(row)<=set(new_ids) for row in vectors.values())

RAW_TOKEN=re.compile(r"[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*",re.IGNORECASE)
def normalized_text(value): return " ".join(unicodedata.normalize("NFKC",str(value)).lower().split())
def canonical_decimal(value):
    rendered=format(Decimal(str(value).replace(",","")).normalize(),"f")
    rendered=rendered.rstrip("0").rstrip(".") if "." in rendered else rendered
    return "0" if rendered in {"","-0"} else rendered
def search_tokens(value):
    text=normalized_text(value); tokens=list(RAW_TOKEN.findall(text))
    for run in re.findall(r"[가-힣](?:[가-힣 ]{0,38}[가-힣])?",text):
        joined=run.replace(" ","")
        for size in (2,3,4):
            tokens.extend("ko"+str(size)+"_"+joined[i:i+size] for i in range(max(0,len(joined)-size+1)))
    consumed=[]
    for match in re.finditer(r"(\d[\d,]*(?:\.\d+)?)\s*만\s*(\d[\d,]*(?:\.\d+)?)\s*천\s*원",text):
        amount=Decimal(match.group(1).replace(",",""))*10000+Decimal(match.group(2).replace(",",""))*1000
        tokens.append("money_krw_"+canonical_decimal(amount)); consumed.append(match.span())
    for match in re.finditer(r"(\d[\d,]*(?:\.\d+)?)\s*(만|천)?\s*원",text):
        if any(left<=match.start() and match.end()<=right for left,right in consumed): continue
        tokens.append("money_krw_"+canonical_decimal(Decimal(match.group(1).replace(",",""))*{"만":10000,"천":1000,None:1}[match.group(2)]))
    for match in re.finditer(r"(\d[\d,]*(?:\.\d+)?)\s*%",text):
        tokens.append("percent_"+canonical_decimal(match.group(1)))
    for match in re.finditer(r"(?:(월|연|년|일)\s*)?(\d[\d,]*(?:\.\d+)?)\s*(회|개월|년|일)",text):
        tokens.append("period_"+(match.group(1) or "none")+"_"+canonical_decimal(match.group(2))+"_"+match.group(3))
    return tokens

documents={identifier:search_tokens(row["retrieval_text"]) for identifier,row in new_by_id.items()}
query_tokens={qid:search_tokens(row["query"]) for qid,row in query_rows.items()}
df=Counter(token for tokens in documents.values() for token in set(tokens))
avg_len=sum(map(len,documents.values()))/len(documents)
def bm25_rank(qid,k1,b):
    scores={}
    for identifier,tokens in documents.items():
        freq=Counter(tokens); score=0.0
        for token in query_tokens[qid]:
            f=freq[token]
            if f:
                idf=math.log(1+(len(documents)-df[token]+.5)/(df[token]+.5))
                score+=idf*f*(k1+1)/(f+k1*(1-b+b*len(tokens)/avg_len))
        scores[identifier]=score
    return sorted(scores,key=lambda identifier:(-scores[identifier],identifier))
def fuse(vector,bm25,vw,bw):
    scores=defaultdict(float)
    for weight,component in ((vw,vector[:50]),(bw,bm25[:50])):
        for rank,identifier in enumerate(component,1): scores[identifier]+=weight/(60+rank)
    top50=sorted(scores,key=lambda identifier:(-scores[identifier],identifier))[:50]
    assert len(top50)==len(set(top50))==50 and set(top50)<=set(vector[:50])|set(bm25[:50])
    for left,right in zip(top50,top50[1:]):
        if scores[left]==scores[right]: assert left<right
    return top50,scores
def relevant(gold):
    return {identifier for identifier,row in new_by_id.items()
      if row["metadata"]["card_key"]==gold["expected_card"]
      and all(normalized_text(term) in normalized_text(row["evidence_text"]) for term in gold["required_terms"])}
def evaluate(gold,ranking):
    cards=[new_by_id[item]["metadata"]["card_key"]==gold["expected_card"] for item in ranking[:5]]
    first=next((i for i,hit in enumerate(cards,1) if hit),None)
    result={"card_hit_at_3":int(any(cards[:3])),"card_mrr_at_5":1/first if first else 0.0}
    if gold["expected_level"]=="card":
        return {**result,"answer_hit_at_3":None,"answer_recall_at_5":None,
         "answer_mrr_at_5":None,"answer_ndcg_at_5":None,"answer_relevant_count":None}
    rel=relevant(gold); assert rel
    hits=[item in rel for item in ranking[:5]]
    first=next((i for i,hit in enumerate(hits,1) if hit),None)
    dcg=sum(hit/math.log2(i+1) for i,hit in enumerate(hits,1))
    ideal=sum(1/math.log2(i+1) for i in range(1,min(5,len(rel))+1))
    return {**result,"answer_hit_at_3":int(any(hits[:3])),"answer_recall_at_5":sum(hits)/len(rel),
     "answer_mrr_at_5":1/first if first else 0.0,"answer_ndcg_at_5":dcg/ideal,
     "answer_relevant_count":len(rel)}
def aggregate(rows):
    result={}
    for metric in ("card_hit_at_3","card_mrr_at_5","answer_hit_at_3","answer_recall_at_5","answer_mrr_at_5","answer_ndcg_at_5"):
        values=[row[metric] for row in rows if row[metric] is not None]
        result[metric]=sum(values)/len(values) if values else None
        result[metric+"_denominator"]=len(values)
    return result

bm25s={(k1,b,qid):bm25_rank(qid,k1,b) for k1 in F2_K1_VALUES for b in F2_B_VALUES for qid in query_rows}
assert all(len(row)==len(set(row))==147 for row in bm25s.values())
ranking_rows=[]; per_query=[]; all_rankings={}
for config in F2_CONFIGS:
    for qid,gold in query_rows.items():
        bm25=bm25s[(config["k1"],config["b"],qid)]
        ranking,scores=fuse(vectors[qid],bm25,config["vector_weight"],config["bm25_weight"])
        all_rankings[(config["configuration"],qid)]=ranking
        provenance={**config,"rrf_k":60,"component_depth":50,"fused_depth":50,"query_id":qid}
        ranking_rows.append({**provenance,
         "vector_top50_chunk_ids":canonical_json(vectors[qid]),
         "bm25_top50_chunk_ids":canonical_json(bm25[:50]),
         "fused_top50_chunk_ids":canonical_json(ranking),
         "fused_top50_scores":canonical_json([scores[item] for item in ranking])})
        per_query.append({**provenance,
         "question_group":"card" if gold["expected_level"]=="card" else "evidence",
         "category":gold["category"],**evaluate(gold,ranking),
         "top5_chunk_ids":canonical_json(ranking[:5]),
         "top5_cards":canonical_json([new_by_id[item]["metadata"]["card_key"] for item in ranking[:5]])})
assert len(ranking_rows)==len(per_query)==810
default_config=next(row["configuration"] for row in F2_CONFIGS
 if (row["k1"],row["b"],row["vector_weight"],row["bm25_weight"])==(1.5,.75,.4,.6))
for qid in query_rows: assert all_rankings[(default_config,qid)]==saved_freeze[qid]["top50_chunk_ids"]

summary=[]
for config in F2_CONFIGS:
    rows=[row for row in per_query if row["configuration"]==config["configuration"]]
    groups={"all":rows,"card":[r for r in rows if r["question_group"]=="card"],
     "evidence":[r for r in rows if r["question_group"]=="evidence"],
     "numeric":[r for r in rows if r["category"]=="numeric_condition"],
     "semantic":[r for r in rows if r["category"]=="semantic"]}
    summary.extend({**config,"group":group,"denominator":len(selected),**aggregate(selected)}
                   for group,selected in groups.items())
assert len(summary)==135
summary_by_key={(row["configuration"],row["group"]):row for row in summary}
for group in ("all","card","evidence","numeric","semantic"):
    produced=summary_by_key[(default_config,group)]
    stored=stored_f1_by_key[("structural_heading_path",group)]
    for metric in ("card_hit_at_3","card_mrr_at_5","answer_hit_at_3","answer_recall_at_5","answer_mrr_at_5","answer_ndcg_at_5"):
        assert produced[metric] is stored[metric] if produced[metric] is None or stored[metric] is None else abs(produced[metric]-stored[metric])<=1e-12
def gate(configuration):
    checks={}
    for group,metrics in F2_GUARDRAILS.items():
        for metric,threshold in metrics.items():
            checks[group+"_"+metric]=summary_by_key[(configuration,group)][metric]>=threshold-1e-12
    return checks
def selection_key(config):
    rows={group:summary_by_key[(config["configuration"],group)] for group in ("evidence","numeric","semantic","card")}
    priority=(rows["evidence"]["answer_hit_at_3"],rows["numeric"]["answer_hit_at_3"],
     rows["evidence"]["answer_mrr_at_5"],rows["numeric"]["answer_mrr_at_5"],
     rows["semantic"]["answer_mrr_at_5"],rows["card"]["card_mrr_at_5"])
    distance=abs(config["k1"]-1.5)+abs(config["b"]-.75)+abs(config["vector_weight"]-.4)+abs(config["bm25_weight"]-.6)
    return tuple(-x for x in priority)+(distance,config["grid_order"])
gate_rows=[]
for config in F2_CONFIGS:
    checks=gate(config["configuration"]); gate_rows.append({**config,**checks,"gate_pass":all(checks.values())})
passing=[config for config in F2_CONFIGS if all(gate(config["configuration"]).values())]
raw_best=min(F2_CONFIGS,key=selection_key); selected=min(passing,key=selection_key) if passing else None
disposition="exploratory_candidate_only" if selected else "retain_existing_operational_prototype"

with F1_PER_QUERY.open(encoding="utf-8",newline="") as handle: f1_rows=list(csv.DictReader(handle))
metric_names=("card_hit_at_3","card_mrr_at_5","answer_hit_at_3","answer_recall_at_5","answer_mrr_at_5","answer_ndcg_at_5")
old_by_query={}
for row in f1_rows:
    if row["configuration"]=="existing_exact_v1_leaf_filtered":
        old_by_query[row["query_id"]]={metric:None if row[metric]=="" else float(row[metric]) for metric in metric_names}
assert len(old_by_query)==30
deltas=[]
for row in per_query:
    result={key:row[key] for key in ("configuration","grid_order","k1","b","vector_weight","bm25_weight","query_id","question_group","category")}
    for metric in metric_names:
        old=old_by_query[row["query_id"]][metric]
        result["baseline_"+metric]=old; result["structural_"+metric]=row[metric]
        result["delta_"+metric]=None if old is None else row[metric]-old
    deltas.append(result)
wlt=[]
for config in F2_CONFIGS:
    rows=[row for row in deltas if row["configuration"]==config["configuration"]]
    groups={"all":rows,"card":[r for r in rows if r["question_group"]=="card"],
     "evidence":[r for r in rows if r["question_group"]=="evidence"],
     "numeric":[r for r in rows if r["category"]=="numeric_condition"],
     "semantic":[r for r in rows if r["category"]=="semantic"]}
    for group,selected_rows in groups.items():
        for metric in metric_names:
            values=[r["delta_"+metric] for r in selected_rows if r["delta_"+metric] is not None]
            if not values: continue
            wins=sum(x>1e-12 for x in values); losses=sum(x< -1e-12 for x in values)
            wlt.append({"configuration":config["configuration"],"grid_order":config["grid_order"],
             "group":group,"metric":metric,"denominator":len(values),"wins":wins,"losses":losses,
             "ties":len(values)-wins-losses,"mean_delta":sum(values)/len(values)})
assert len(deltas)==810 and len(wlt)==702
assert all(r["wins"]+r["losses"]+r["ties"]==r["denominator"] for r in wlt)
assert all(value is None or 0<=value<=1 for row in per_query for key,value in row.items() if key in metric_names)

write_csv(OUT/"followup2_rankings.csv",ranking_rows,list(ranking_rows[0]))
write_csv(OUT/"followup2_per_query.csv",per_query,list(per_query[0]))
write_csv(OUT/"followup2_summary.csv",summary,list(summary[0]))
write_csv(OUT/"followup2_paired_deltas.csv",deltas,list(deltas[0]))
write_csv(OUT/"followup2_wlt.csv",wlt,list(wlt[0]))
wall_seconds=time.perf_counter()-f2_wall_start; cpu_seconds=time.process_time()-f2_cpu_start
rss_end=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
resources={"measurement_scope":"append_only_followup2_cells_fresh_kernel","existing_notebook_cells_reexecuted":False,
 "wall_seconds":wall_seconds,"cpu_seconds":cpu_seconds,"process_peak_rss_mib":rss_end/1024,
 "process_peak_rss_start_mib":f2_rss_start/1024,"phase_incremental_peak_rss_mib":max(0,rss_end-f2_rss_start)/1024,
 "peak_rss_limitation":"ru_maxrss is process-wide, not isolated phase allocation",**F2_CONTRACT["execution"],
 "unmeasured":["energy","per-query latency"]}
decision={"experiment_type":"27_config_development_grid_exploration","formal_promotion_eligible":False,
 "guardrails":F2_GUARDRAILS,"passing_configuration_count":len(passing),"all_configurations_gate":gate_rows,
 "raw_best_configuration":raw_best,"selected_exploratory_configuration":selected,
 "disposition":disposition,"current_operational_prototype":"existing_exact_v1_leaf_filtered",
 "default_structural_configuration":default_config,"selection_priority":F2_PRIORITY,
 "limitations":["Same 30 development queries were searched over 27 configurations.",
 "Recall and nDCG are diagnostic because corpus/relevance denominators differ.",
 "Historical notebook16 reproduction remains incomplete at 30/20/28/19.",
 "No configuration is eligible for formal operations or broader-data promotion."]}
write_json(OUT/"followup2_resources.json",resources); write_json(OUT/"followup2_decision.json",decision)
write_json(OUT/"followup2_summary.json",{"contract_raw_sha256":raw_sha256(OUT/"followup2_contract.json"),
 "summaries":summary,"gate_results":gate_rows,"decision":decision,"resources":resources,
 "metric_guide_ko":{"Card Hit@3":"상위 3개에 기대 카드가 있는 질의 비율",
 "Card MRR@5":"상위 5개에서 기대 카드 첫 순위 역수 평균",
 "Answer Hit@3":"상위 3개에 카드와 필수 용어를 포함한 청크가 있는 질의 비율",
 "Answer MRR@5":"상위 5개에서 해당 청크 첫 순위 역수 평균",
 "Recall/nDCG":"청크 분모 변화에 민감한 진단값"}})
input_hashes_after={str(path.relative_to(ROOT)):raw_sha256(path) for path in READ_ONLY_INPUTS}
assert input_hashes_after==input_hashes_before
integrity2={"status":"PASS","execution_mode":F2_CONTRACT["execution_mode"],
 "existing_notebook_cells_reexecuted":False,"configuration_count":27,"query_count":30,
 "ranking_rows":810,"per_query_rows":810,"summary_rows":135,"paired_delta_rows":810,"wlt_rows":702,
 "component_top50_saved":True,"fused_top50_unique_and_union_contained":True,
 "score_tie_break_verified":True,"baseline_followup1_exact":True,"default_structural_exact":True,
 "metric_ranges":True,"input_hashes_before":input_hashes_before,"input_hashes_after":input_hashes_after,
 "inputs_unchanged":True,"cache":{"new_documents":147,"combined_old_items":357,"dimension":1536,
 "dtype":"float32","finite":True},"execution":F2_CONTRACT["execution"],
 "passing_configuration_count":len(passing),"disposition":disposition}
write_json(OUT/"followup2_integrity.json",integrity2)

readme_path=OUT/"README.md"; readme=readme_path.read_text(encoding="utf-8")
marker="## Follow-up 2 — 새 구조 BM25/RRF 27조합 탐색"
if marker in readme: readme=readme.split(marker)[0].rstrip()
readme+=f"""

{marker}

기존 notebook 셀은 이번 실행에서 재실행하지 않았습니다. self-contained Follow-up 2 셀만 별도 fresh kernel에서 실행해 저장된 chunks/cache/query/Follow-up 1을 직접 읽었습니다.

27개 조합을 모두 저장했고 새 API·network·embedding·GPU·Chroma query·검색 index 변경은 0, API 비용은 $0입니다. Recall/nDCG는 진단값입니다.

gate 통과 조합 수는 {len(passing)}개이고 disposition은 {disposition}입니다. 통과해도 개발 질의 27-grid 탐색이므로 정식 운영 승격은 불가능합니다. historical notebook16 재현 한계 30/20/28/19도 유지됩니다.
"""
readme_path.write_text(readme.rstrip()+"\n",encoding="utf-8")
excluded={"run_manifest.json","integrity.json"}
output_paths=sorted(path for path in OUT.rglob("*") if path.is_file() and path.name not in excluded)
manifest={"experiment_name":"구조 기반 청킹 + 제목 경로 검색문","execution_mode":F2_CONTRACT["execution_mode"],
 "existing_notebook_cells_reexecuted":False,"input_raw_sha256":input_hashes_before,
 "output_raw_sha256":{str(path.relative_to(OUT)):raw_sha256(path) for path in output_paths},
 "notebook_raw_sha256":None,"notebook_hash_finalization":"finalized_after_append_only_execution",
 "self_hash_policy":"run_manifest.json and integrity.json excluded"}
write_json(OUT/"run_manifest.json",manifest)
integrity=json.loads((OUT/"integrity.json").read_text(encoding="utf-8"))
integrity["followup2"]=integrity2; integrity["last_execution_mode"]=F2_CONTRACT["execution_mode"]
integrity["existing_notebook_cells_reexecuted"]=False; integrity["notebook_raw_sha256"]=None
integrity["notebook_hash_finalization"]="finalized_after_append_only_execution"
integrity["run_manifest_raw_sha256"]=raw_sha256(OUT/"run_manifest.json")
write_json(OUT/"integrity.json",integrity)
assert {str(path.relative_to(ROOT)):raw_sha256(path) for path in READ_ONLY_INPUTS}==input_hashes_before
print({"followup2":"PASS","passing":len(passing),"raw_best":raw_best["configuration"],
 "selected":None if selected is None else selected["configuration"],"disposition":disposition,
 "wall_seconds":wall_seconds})


{'followup2': 'PASS', 'passing': 0, 'raw_best': 'k1_1.5_b_0.75_vector_0.4_bm25_0.6', 'selected': None, 'disposition': 'retain_existing_operational_prototype', 'wall_seconds': 2.209449542919174}
